In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: IMPORTS & CONFIGURATION
# All libraries loaded once here. Change TARGET_MONTH each month —
# every path, name, and link is derived from it; nothing else changes.
# ═══════════════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

import pandas as pd
import re
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import FixedLocator
from collections import Counter
from datetime import datetime
# ── ✏ CHANGE THIS EVERY MONTH — everything else derives from it ──
TARGET_MONTH = pd.Period("2026-06", freq="M")
REPORT_BASE_DIR = r"C:\Users\shreeharsha.r\OneDrive - Apollo Tyres Limited\Patent_Patseer\Patent_Monthly_report"
AUTHOR       = "Shreeharsha Ramaswamy"
REPORT_TITLE = "Patents – Monthly Analysis Report"
# Subtle methodology note shown at the end of the Executive Summary. Keep
# it while the expanded-query counts are still new vs. older reports; set
# to "" to remove once the new baseline is established.
SCOPE_CHANGE_NOTE = (
    "Note: search parameters were expanded this cycle to improve coverage of "
    "tire design and construction patents, so patent counts are not directly "
    "comparable with earlier months' reports."
)
# Hard monthly ceiling on patents that get PDF drawings + folder links.
# This is YOUR PDF-handling capacity, not a data-driven number — the
# notable set is ranked by Patent Interest Score and truncated to this,
# so the workload can never balloon just because a month is busy.
MAX_NOTABLE_PATENTS = 25
# Max rows shown in each Section-4 thematic table (Cross-domain, AI/ML,
# Sustainability, Smart/Connected). Full lists always remain in the
# classified Excel; this only trims the Word report for readability.
SECTION4_MAX_ROWS_PER_TABLE = 10
# Section-5 "one page per card" rule: Abstract and Independent Claims are
# the only free-length fields on a card, so budgeting their character
# counts keeps each card within ~one page. Truncated text ends at a
# sentence boundary with a note; the full text always stays in the
# classified Excel. Lower these if cards still run over a page in Word;
# raise them for more detail per card.
SECTION5_ABSTRACT_MAX_CHARS = 650
SECTION5_CLAIMS_MAX_CHARS    = 1100
# Start every Section-5 patent card on a fresh page (one card per page).
SECTION5_ONE_CARD_PER_PAGE   = True

# ── Auto-derived — do not edit ────────────────────────────────
MONTH_NAME   = TARGET_MONTH.strftime("%B %Y")          # "June 2026"
MONTH_FOLDER = TARGET_MONTH.strftime("%B%Y")            # "June2026"
YEAR         = TARGET_MONTH.year
MONTH_NUMBER = TARGET_MONTH.month
FILE_DIR     = os.path.join(REPORT_BASE_DIR, MONTH_FOLDER)
FILE_NAME    = f"Patent_report_{MONTH_FOLDER}.xlsx"
FILE_PATH    = os.path.join(FILE_DIR, FILE_NAME)

print("✔ All imports loaded successfully.")
print(f"  Target month : {MONTH_NAME}")
print(f"  File         : {FILE_PATH}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: LOAD EXCEL FILE + DATA QUALITY DIAGNOSTICS
# Loads the PatSeer export and prints a quick quality summary.
# ═══════════════════════════════════════════════════════════════

df = pd.read_excel(FILE_PATH)

print("✔ Hyperlinks stripped from all text columns.")
print(f"✔ File loaded successfully.")
print(f"  Rows    : {len(df)}")
print(f"  Columns : {len(df.columns)}")
print(f"\n=== MISSING VALUES ===")
print(df.isnull().sum().to_string())
print(f"\n=== DUPLICATE PATENT NUMBERS ===")
print(f"  Total duplicates: {df.duplicated(subset='Record Number').sum()}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2B: EXTRACT HYPERLINKS FROM PDF LINK COLUMN
# ════════════════════════════════════════════════════════════════

import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows

print("Extracting hyperlinks from PDF Link column...")

# Load workbook with openpyxl
wb = openpyxl.load_workbook(FILE_PATH)
ws = wb.active

# Find PDF Link column index (assumes first row is header)
headers = [cell.value for cell in ws[1]]
try:
    pdf_col_idx = headers.index('PDF Link') + 1  # openpyxl is 1-indexed
except ValueError:
    print("⚠ 'PDF Link' column not found — skipping hyperlink extraction")
    pdf_col_idx = None

# Extract hyperlinks if column exists
if pdf_col_idx:
    extracted_links = []
    
    for row_idx in range(2, ws.max_row + 1):  # start from row 2 (skip header)
        cell = ws.cell(row=row_idx, column=pdf_col_idx)
        
        if cell.hyperlink and cell.hyperlink.target:
            # Cell has hyperlink — extract URL
            url = cell.hyperlink.target
        else:
            # No hyperlink — keep cell text or blank
            url = cell.value if cell.value else ''
        
        extracted_links.append(url)
    
    # Add new column to dataframe
    df['PDF_Link_URL'] = extracted_links
    
    filled = df['PDF_Link_URL'].astype(str).str.startswith('http').sum()
    print(f"✔ Hyperlinks extracted: {filled}/{len(df)} valid URLs")
    print(f"  Sample: {df['PDF_Link_URL'].head(2).tolist()}")
else:
    df['PDF_Link_URL'] = ''
    print("⚠ No hyperlinks extracted — 'PDF_Link_URL' column created empty")

print()

# ── PatSeer PDF Link lookup ───────────────────────────────────
PATSEER_BASE = "https://app.patseer.com/#/search/result?q="

if 'pdf_links' not in dir():
    pdf_links = {str(r).strip(): PATSEER_BASE + str(r).strip()
                 for r in df['Record Number']}
    print("⚠ pdf_links rebuilt from record numbers (Cell 2B may not have run)")
else:
    print(f"✔ pdf_links ready: {len(pdf_links)} entries")
    print(f"  Sample: {list(pdf_links.items())[:2]}")

In [ ]:
import os
import re
import glob
import pandas as pd

# ================= SETTINGS =================
PDF_SUBFOLDER = "Downloaded_Patent_PDFs"
RECORD_COL = "Record Number"
# ===========================================

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = re.sub(r"[^A-Z0-9]", "", x)
    return x

def normalize_patent_number(x):
    """
    General patent number normalization.

    Handles cases like:
    US20260092169A1 -> US2026092169A1
    EP074263A2      -> EP74263A2
    DE0102024136400B4 -> DE102024136400B4

    Logic:
    - remove symbols/spaces
    - split into:
        country letters + numeric body + kind code
    - remove leading zeros from numeric body carefully
    """
    x = clean_text(x)

    if not x:
        return ""

    # Country code + number body + kind code
    # Example: US20260092169A1
    m = re.match(r"^([A-Z]{2})(\d+)([A-Z]\d?)$", x)

    if m:
        country, number_body, kind = m.groups()

        # Remove leading zeros in number body
        number_body = number_body.lstrip("0")

        return f"{country}{number_body}{kind}"

    return x

def generate_patent_variants(x):
    """
    Creates multiple normalized variants to improve matching.
    """
    x_clean = clean_text(x)
    x_norm = normalize_patent_number(x)

    variants = set()
    variants.add(x_clean)
    variants.add(x_norm)

    # Also remove all zeros after country code cautiously
    m = re.match(r"^([A-Z]{2})(\d+)([A-Z]\d?)$", x_clean)
    if m:
        country, number_body, kind = m.groups()

        variants.add(f"{country}{number_body.lstrip('0')}{kind}")

        # Remove zeros immediately after first 4-digit year if present
        # Useful for publication numbers: US20260092169A1 -> US2026092169A1
        year_match = re.match(r"^(\d{4})(0+)(\d+)$", number_body)
        if year_match:
            year, zeros, rest = year_match.groups()
            variants.add(f"{country}{year}{rest}{kind}")

    return [v for v in variants if v]

# -------- collect PDFs --------
pdf_files = sorted(glob.glob(os.path.join(FILE_DIR, PDF_SUBFOLDER, "*.pdf")))

pdf_index = []
for path in pdf_files:
    fname = os.path.basename(path)
    fname_base = os.path.splitext(fname)[0]

    pdf_index.append({
        "path": path,
        "filename": fname,
        "variants": generate_patent_variants(fname_base),
        "clean": clean_text(fname_base)
    })

def find_local_pdf(record_number):
    row_variants = generate_patent_variants(record_number)

    if not row_variants:
        return "", "Missing"

    # 1. Exact variant match
    matches = []
    for item in pdf_index:
        if any(rv == pv for rv in row_variants for pv in item["variants"]):
            matches.append(item)

    if len(matches) == 1:
        return matches[0]["path"], "Found - exact normalized variant"

    if len(matches) > 1:
        return "", f"Ambiguous - exact variant matched {len(matches)} files"

    # 2. Contains match
    matches = []
    for item in pdf_index:
        for rv in row_variants:
            if rv in item["clean"] or item["clean"] in rv:
                matches.append(item)
                break

    matches = list({m["path"]: m for m in matches}.values())

    if len(matches) == 1:
        return matches[0]["path"], "Found - contains normalized variant"

    if len(matches) > 1:
        return "", f"Ambiguous - contains matched {len(matches)} files"

    # 3. Last 8 unique fallback
    key = max(row_variants, key=len)
    for n in [8, 6, 4]:
        suffix = key[-n:]

        matches = []
        for item in pdf_index:
            if any(suffix in pv for pv in item["variants"]):
                matches.append(item)

        matches = list({m["path"]: m for m in matches}.values())

        if len(matches) == 1:
            return matches[0]["path"], f"Found - last{n} unique fallback"

        if len(matches) > 1:
            return "", f"Ambiguous - last{n} matched {len(matches)} files"

    return "", "Missing"

# -------- apply mapping --------
results = df[RECORD_COL].apply(find_local_pdf)

df["Local_PDF_Path"] = results.apply(lambda x: x[0])
df["PDF_Match_Status"] = results.apply(lambda x: x[1])

df["PDF_Status"] = df["Local_PDF_Path"].apply(
    lambda x: "Found" if isinstance(x, str) and x.strip() else "Missing"
)

df["Local_PDF_Relative_Path"] = df["Local_PDF_Path"].apply(
    lambda x: os.path.relpath(x, ".") if isinstance(x, str) and x.strip() else ""
)

df["Open_Local_PDF"] = df["Local_PDF_Relative_Path"].apply(
    lambda x: f'=HYPERLINK("{x}", "Open PDF")' if x else ""
)

print("PDF files found:", len(pdf_files))

print("\nPDF Status Summary:")
print(df["PDF_Status"].value_counts())
print(
    "  NOTE: PDF drawing embedding is scoped to notable patents only (see "
    "Cell 6F), so a large 'Missing' count here is expected at full monthly "
    "volume — it does not need to reach 0. Typical workflow: run once to "
    "get Notable_Patents_PDFs_Needed_<month>.xlsx, bulk-download PDFs for "
    "just that list, then re-run so those get matched."
)

print("\nMatch Status Summary:")
print(df["PDF_Match_Status"].value_counts())

print("\nStill missing / ambiguous (showing up to 30):")
display_cols = [RECORD_COL, "PDF_Match_Status"]
if "Title" in df.columns:
    display_cols.append("Title")

display(df[df["PDF_Status"] == "Missing"][display_cols].head(30))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: DATA CLEANING - SAFE VERSION
# Does NOT remove rows unless they are fully empty.
# Flags missing abstracts and duplicate record numbers instead.
# ═══════════════════════════════════════════════════════════════

print("─── Starting Data Cleaning ───────────────────────────────")
print(f"Rows before cleaning : {len(df)}")

# 1. Standardize column names — strip spaces and newlines
df.columns = df.columns.str.strip().str.replace('\n', ' ', regex=False)

# 2. Drop only fully empty rows
before_empty = len(df)
df = df.dropna(how='all')
empty_removed = before_empty - len(df)
print(f"Fully empty rows removed: {empty_removed}")

# 3. DO NOT drop rows with missing Abstract
# Instead, flag them
if "Abstract" in df.columns:
    df["Missing_Abstract_Flag"] = df["Abstract"].isna() | (df["Abstract"].astype(str).str.strip() == "")
    print("Rows with missing Abstract:", df["Missing_Abstract_Flag"].sum())
else:
    print("⚠ Abstract column not found")

# 4. Strip whitespace from selected text columns safely
text_cols = [
    'Assignee', 'Title', 'Abstract', 'Claims',
    'Independent Claims', 'Description',
    'Problem Being Solved (AI Sum.)',
    'Method Used (AI Sum.)',
    'Advantages (AI Sum.)',
    'Record Number',
    'PDF Link'
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# 5. Parse date column to datetime
if 'Publication/Issue Date' in df.columns:
    df['Publication/Issue Date'] = pd.to_datetime(
        df['Publication/Issue Date'], errors='coerce'
    )

# 6. Fill remaining missing values safely
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('').astype(str).str.strip()
    elif pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(0)

# 7. DO NOT remove duplicates automatically
# Instead, flag duplicate Record Numbers
if "Record Number" in df.columns:
    df["Duplicate_Record_Flag"] = df.duplicated(subset=["Record Number"], keep=False)
    duplicate_count = df["Duplicate_Record_Flag"].sum()
    print(f"Duplicate rows flagged: {duplicate_count}")
else:
    print("⚠ Record Number column not found")

# 8. Reset index
df = df.reset_index(drop=True)

# 9. Create lowercase shadow columns for keyword matching only
for col in ['Title', 'Abstract', 'Claims', 'Independent Claims']:
    if col in df.columns:
        df[col + '_lower'] = df[col].fillna("").astype(str).str.lower()

# 10. Safe date range print
if 'Publication/Issue Date' in df.columns and df['Publication/Issue Date'].notna().any():
    print(f"Date range           : {df['Publication/Issue Date'].min().date()} "
          f"to {df['Publication/Issue Date'].max().date()}")
else:
    print("Date range           : No valid dates found")

print(f"Rows after cleaning  : {len(df)}")
print("─── Cleaning Complete ────────────────────────────────────")

# ── Stale/wrong-month guard ────────────────────────────────────
# Prevents the class of bug where FILE_DIR/FILE_NAME/MONTH_NAME are edited
# after the last run (or point at the wrong export) and the report silently
# describes a different month than the data it was built from.
if 'Publication/Issue Date' in df.columns and df['Publication/Issue Date'].notna().any():
    pub_periods = df['Publication/Issue Date'].dropna().dt.to_period('M')
    in_target_month = (pub_periods == TARGET_MONTH).sum()
    out_of_target_month = (pub_periods != TARGET_MONTH).sum()

    print(f"\n─── Month Consistency Check ──────────────────────────────")
    print(f"Target report month  : {TARGET_MONTH}")
    print(f"Records in target month     : {in_target_month}")
    print(f"Records outside target month: {out_of_target_month}")

    if in_target_month == 0:
        raise ValueError(
            f"None of the {len(df)} records fall in {TARGET_MONTH}. "
            f"Actual publication range: {df['Publication/Issue Date'].min().date()} "
            f"to {df['Publication/Issue Date'].max().date()}. "
            f"Check TARGET_MONTH / FILE_DIR / FILE_NAME in Cell 1 — "
            f"you are likely pointed at the wrong month's export."
        )
    if out_of_target_month > 0:
        print(
            f"⚠ {out_of_target_month} record(s) fall outside {TARGET_MONTH}. "
            f"Report will proceed but double-check the source export."
        )
    print("─────────────────────────────────────────────────────────")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: ALL HELPER FUNCTIONS
# Defined once here and reused across all downstream cells.
# Do not modify unless changing core logic.
# ═══════════════════════════════════════════════════════════════

# ── Segment Classification ────────────────────────────────────
def _word_positions(text):
    """Returns (word_list, char_start_of_each_word) for proximity checks."""
    words = []
    starts = []
    for m in re.finditer(r"\S+", text):
        words.append(m.group(0))
        starts.append(m.start())
    return words, starts


def _token_near_tire_terms(text, token, words, starts, window=6):
    """
    True if a whole-word match of `token` occurs within `window` words
    of a tire/tyre term. Used to gate generic/ambiguous single-word
    keywords (e.g. 'bus', 'car', 'race') that otherwise false-positive
    on unrelated words such as 'bushing' or 'scarce'.
    """
    tire_word_idx = [i for i, w in enumerate(words)
                      if re.sub(r"[^a-z]", "", w) in TIRE_TERMS]
    if not tire_word_idx:
        return False
    pattern = re.compile(r'\b' + re.escape(token) + r'\b')
    for m in pattern.finditer(text):
        word_idx = next((i for i, s in enumerate(starts) if s >= m.start()), len(starts) - 1)
        if any(abs(word_idx - ti) <= window for ti in tire_word_idx):
            return True
    return False


def classify_segment(abstract, claims, title, safe_phrases, risky_tokens):
    """
    Matches Title (weighted 2x) + Abstract + Claims against segment
    definitions using two tiers of evidence:
      1. High-precision multi-word phrases (SEGMENT_SAFE_PHRASES) — matched
         anywhere, whole-phrase, low false-positive risk.
      2. Generic single-word tokens (SEGMENT_RISKY_SINGLE_TOKENS) — only
         counted as evidence if they appear as a whole word within a small
         window of an explicit tire/tyre mention, and always word-boundaried
         so 'bus' cannot match inside 'bushing', 'car' inside 'scarce', etc.
    Returns pipe-separated matched segments, or 'General' if no match found.
    """
    combined = (str(title) + ' ' + str(title) + ' ' +
                str(abstract) + ' ' + str(claims)).lower()
    words, starts = _word_positions(combined)
    matched = []
    for segment, phrases in safe_phrases.items():
        if any(phrase in combined for phrase in phrases):
            matched.append(segment)
            continue
        for token in risky_tokens.get(segment, []):
            if _token_near_tire_terms(combined, token, words, starts):
                matched.append(segment)
                break
    return ' | '.join(matched) if matched else 'General'

# ── Assignee Cleaning & Normalization ─────────────────────────
def clean_assignee(text):
    """Lowercases and removes punctuation from assignee text."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def match_company(cleaned_text, company_list, raw_text=""):
    """
    Matches assignee text against known tire companies, in three tiers:
      1. ASSIGNEE_ALIASES — literal substring match on lightly-normalized
         raw text (lowercased, not ASCII-stripped). Needed because
         clean_assignee() strips all non a-z0-9 characters, which silently
         destroys non-Latin-script names (e.g. a Korean-language Hankook
         filing) and doesn't bridge legal-entity names to brand names
         (e.g. 'Cheng Shin Rubber Industry Co Ltd' -> Maxxis).
      2. TIRE_COMPANIES — word-set match on the ASCII-cleaned text
         (original approach, kept for backward compatibility).
      3. Fallback: if the raw text contains 'tire'/'tyre'/'rubber' as a
         whole word but matched no known company, tag it as
         'Other Tire/Rubber Manufacturer' rather than lumping it in with
         unrelated assignees (universities, chemical suppliers, OEMs,
         research institutes) under a single undifferentiated 'Others'.
    Returns Title Case company name, 'Other Tire/Rubber Manufacturer',
    or 'Others'.
    """
    raw_lower = str(raw_text).lower()
    for alias, canonical in ASSIGNEE_ALIASES.items():
        if alias in raw_lower:
            return canonical

    if cleaned_text:
        words = set(cleaned_text.split())
        for company in company_list:
            if all(w in words for w in company.split()):
                return company.title()

    if re.search(r'\b(tire|tyre|rubber)\b', raw_lower):
        return "Other Tire/Rubber Manufacturer"

    return "Others"


# ── Taxonomy Categorization ───────────────────────────────────
def categorize_patent(text, taxonomy):
    """
    Scans text for matches against a taxonomy dictionary.
    Input text must already be lowercased (shadow column).
    Returns (matched_labels list, evidence dict).
    """
    if not text or text.strip() == '':
        return [], {}
    matched_labels = []
    evidence = {}
    for category, keywords in taxonomy.items():
        for keyword in keywords:
            if keyword in text:
                matched_labels.append(category)
                evidence[category] = keyword
                break
    return matched_labels, evidence

def apply_categorization(abstract, claims, taxonomy):
    """Combines abstract + claims and returns pipe-separated matched labels."""
    combined = str(abstract) + ' ' + str(claims)
    labels, _ = categorize_patent(combined, taxonomy)
    return ' | '.join(labels) if labels else 'None'

def apply_evidence(abstract, claims, taxonomy):
    """Returns pipe-separated evidence keywords for matched categories."""
    combined = str(abstract) + ' ' + str(claims)
    _, evidence = categorize_patent(combined, taxonomy)
    return ' | '.join([f'{k}: "{v}"' for k, v in evidence.items()]) if evidence else 'None'


# ── Label Counter (for charts & tables) ───────────────────────
def count_labels(series):
    """
    Explodes pipe-separated multi-label cells and counts each label.
    Excludes 'None' and empty entries. Returns sorted DataFrame.
    """
    all_labels = []
    for cell in series:
        labels = [l.strip() for l in str(cell).split('|')
                  if l.strip() not in ('None', '')]
        all_labels.extend(labels)
    counts = Counter(all_labels)
    return pd.DataFrame(counts.most_common(), columns=['Label', 'Count'])


# ── AI/ML Detection ───────────────────────────────────────────
AI_KEYWORDS = [
    'machine learning', 'deep learning', 'neural network',
    'artificial intelligence', 'fuzzy logic', 'data analytics',
    'big data', 'predictive model', 'algorithm', 'robotics',
    'computer vision', 'natural language', 'ml model'
]

def detect_aiml(text):
    """Checks lowercase text for AI/ML keywords. Returns matched list or 'None'."""
    matched = [kw for kw in AI_KEYWORDS if kw in str(text)]
    return ' | '.join(matched) if matched else 'None'

def add_bookmark(paragraph, bookmark_name):
    """Adds a Word bookmark to a paragraph."""
    tag   = paragraph._p
    start = OxmlElement('w:bookmarkStart')
    start.set(qn('w:id'),   str(abs(hash(bookmark_name)) % 100000))
    start.set(qn('w:name'), bookmark_name)
    end   = OxmlElement('w:bookmarkEnd')
    end.set(qn('w:id'), str(abs(hash(bookmark_name)) % 100000))
    tag.insert(0, start)
    tag.append(end)

def add_internal_hyperlink(cell, display_text, bookmark_name):
    """
    Adds a clickable link inside a table cell that jumps
    to a bookmark elsewhere in the same document.
    """
    paragraph = cell.paragraphs[0]
    paragraph.clear()

    hyperlink = OxmlElement('w:hyperlink')
    hyperlink.set(qn('w:anchor'), bookmark_name)   # internal anchor

    run_elem   = OxmlElement('w:r')
    rPr        = OxmlElement('w:rPr')

    color_elem = OxmlElement('w:color')
    color_elem.set(qn('w:val'), '0563C1')
    rPr.append(color_elem)

    u_elem = OxmlElement('w:u')
    u_elem.set(qn('w:val'), 'single')
    rPr.append(u_elem)

    sz_elem = OxmlElement('w:sz')
    sz_elem.set(qn('w:val'), '16')
    rPr.append(sz_elem)

    run_elem.append(rPr)
    t_elem      = OxmlElement('w:t')
    t_elem.text = display_text
    run_elem.append(t_elem)
    hyperlink.append(run_elem)
    paragraph._p.append(hyperlink)

print("✔ All helper functions defined successfully.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: KEYWORD DICTIONARIES (REVISED)
# Edit keywords here only. No changes needed in any other cell.
# # Updates in this revision (segments):
# 1) Added SAFE_PHRASE overrides (high precision phrases)
# 2) Added RISKY_SINGLE_TOKENS (generic words; only count if near tire/tyre)
# 3) Keeps your original SEGMENT_KEYWORDS for backward compatibility,
#    but you SHOULD use SAFE_PHRASES + RISKY_SINGLE_TOKENS in the matcher.
# ═══════════════════════════════════════════════════════════════

SEGMENT_KEYWORDS = {
    # Keep as a master list (backward compatible with your notebook),
    # but for segmentation prefer SAFE_PHRASES + RISKY_SINGLE_TOKENS.
    'TBR': [
        'truck', 'bus', 'truck and bus', 'tbr', 'commercial vehicle',
        'heavy vehicle', 'truck tire', 'bus tire', 'radial truck',
        'load range', 'long haul', 'regional haul', 'steer axle',
        'drive axle', 'trailer axle', 'lug tire', 'rib tire',
        'heavy duty truck', 'coach', 'lorry'
    ],
    'PCR': [
        'passenger car', 'pcr', 'passenger tire', 'passenger vehicle',
        'automobile tire', 'car tire', 'sedan', 'hatchback', 'suv tire',
        'light vehicle', 'passenger radial', 'uhp', 'ultra high performance',
        'run flat', 'run-flat', 'self-sealing tire'
    ],
    'Off_Highway': [
        'off highway', 'off-highway', 'otr', 'off the road',
        'earthmover', 'mining tire', 'construction tire',
        'agricultural tire', 'farm tire', 'tractor tire',
        'loader tire', 'grader tire', 'bulldozer', 'excavator',
        'industrial tire', 'forklift', 'solid tire', 'skid steer'
    ],
    'Two_Wheeler': [
        'motorcycle', 'motorbike', 'two wheeler', 'two-wheeler',
        'scooter', 'moped', 'bicycle tire', 'cycle tire',
        'motor cycle tire', 'bike tire', 'motorcycle tire'
    ],
    'Specialty': [
        'aircraft tire', 'aviation tire', 'airplane tire',
        'race tire', 'racing tire', 'track tire',
        'airless tire', 'non-pneumatic', 'non pneumatic',
        'flat free tire', 'solid rubber tire'
    ]
}

# High-precision phrases: these can match anywhere in Abstract/Claims
# (case-insensitive) with minimal false positives.
SEGMENT_SAFE_PHRASES = {
    'TBR': [
        'truck tire', 'truck tyres', 'bus tire', 'bus tyres',
        'truck and bus', 'tbr', 'commercial vehicle tire', 'commercial vehicle tyre',
        'heavy duty truck', 'steer axle', 'drive axle', 'trailer axle',
        'long haul', 'regional haul', 'load range', 'lug tire', 'rib tire'
    ],
    'PCR': [
        'passenger car', 'passenger tire', 'passenger tyre', 'pcr',
        'car tire', 'car tyre', 'automobile tire', 'automobile tyre',
        'passenger radial', 'uhp', 'ultra high performance',
        'run flat', 'run-flat', 'self-sealing tire', 'self sealing tire', 'suv tire'
    ],
    'Off_Highway': [
        'off highway', 'off-highway', 'off the road', 'otr',
        'mining tire', 'mining tyre', 'construction tire', 'construction tyre',
        'agricultural tire', 'agricultural tyre', 'tractor tire', 'tractor tyre',
        'loader tire', 'grader tire', 'industrial tire', 'forklift tire',
        'skid steer', 'solid tire', 'solid tyre'
    ],
    'Two_Wheeler': [
        'two wheeler', 'two-wheeler', 'motorcycle tire', 'motorcycle tyre',
        'motorbike tire', 'scooter tire', 'moped tire', 'bicycle tire', 'cycle tire',
        'bike tire'
    ],
    'Specialty': [
        'aircraft tire', 'aircraft tyre', 'aviation tire', 'airplane tire',
        'race tire', 'racing tire', 'track tire',
        'airless tire', 'airless tyre',
        'non-pneumatic', 'non pneumatic',
        'flat free tire', 'flat-free tire', 'flat free tyre',
        'solid rubber tire'
    ]
}

# Generic single tokens (high false-positive risk):
# Only count these if within a small word-window of "tire" or "tyre".
SEGMENT_RISKY_SINGLE_TOKENS = {
    'TBR': ['truck', 'bus', 'coach', 'lorry', 'commercial', 'heavy'],
    'PCR': ['car', 'automobile', 'passenger', 'sedan', 'hatchback', 'suv', 'light'],
    'Off_Highway': ['industrial', 'construction', 'tractor', 'forklift', 'earthmover', 'mining',
                    'excavator', 'bulldozer', 'loader', 'grader', 'farm', 'agricultural'],
    'Two_Wheeler': ['motorcycle', 'motorbike', 'scooter', 'moped', 'bike', 'bicycle', 'cycle'],
    'Specialty': ['aircraft', 'aviation', 'airplane', 'race', 'racing', 'track', 'airless', 'solid']
}

# Optional: global tire terms (useful if you implement "near tire/tyre" checks)
TIRE_TERMS = ['tire', 'tyre', 'tires', 'tyres']

# ──────────────────────────────────────────────────────────────
# OTHER DICTIONARIES (UNCHANGED)
# ──────────────────────────────────────────────────────────────

TIRE_COMPANIES = [
    "michelin", "bridgestone", "goodyear", "continental", "pirelli",
    "sumitomo", "yokohama", "hankook", "toyo", "cooper", "nokian",
    "apollo", "ceat", "mrf", "dunlop", "falken", "kumho", "maxxis",
    "nitto", "nexen", "giti", "sailun", "trelleborg", "hutchinson",
    "lanxess", "exxon", "basf", "solvay", "cabot", "evonik"
]

# Substring aliases checked on lightly-normalized (lowercased, not
# ASCII-stripped) raw assignee text, BEFORE the ASCII-cleaned word-set
# match above. Seeded from real "Others" entries found in the June 2026
# export — not guessed:
#   - Hankook's Korean-script legal name was 100% missed by the Latin
#     keyword list above (clean_assignee strips non a-z0-9 characters,
#     turning any Korean/Chinese/Japanese-script name into an empty
#     string) — 31 records, the single biggest miss in that export.
#   - Cheng Shin Rubber (Maxxis's manufacturing parent) files under its
#     legal name, which shares no words with the 'maxxis' keyword.
#   - ZC Rubber (Zhongce) and Linglong are top-15-by-volume global tire
#     manufacturers that simply weren't in the original list.
ASSIGNEE_ALIASES = {
    '한국타이어': 'Hankook',
    'cheng shin rubber': 'Maxxis (Cheng Shin Rubber)',
    'zhongce rubber': 'ZC Rubber (Zhongce)',
    'linglong': 'Linglong',
}

PERFORMANCE_TARGETS = {
    'Low_RR': [
        'rolling resistance', 'low rolling resistance', 'rrc',
        'fuel economy', 'fuel-efficient', 'fuel efficiency',
        'low hysteresis', 'heat build-up', 'heat buildup',
        'energy loss', 'tan delta', 'tan δ', 'energy consumption'
    ],
    'Wet_Grip': [
        'wet traction', 'wet grip', 'wet braking',
        'wet braking performance', 'skid resistance',
        'hydroplaning', 'aquaplaning', 'water drainage', 'grip on wet'
    ],
    'Snow_Ice': [
        'snow traction', 'snow performance', 'ice traction',
        'ice performance', 'winter tire', 'all-season tire',
        'snow grip', 'ice grip', 'slush', 'cold weather'
    ],
    'Wear_Mileage': [
        'wear resistance', 'abrasion', 'tire life', 'mileage',
        'longevity', 'wear performance', 'wear indicator', 'tread life'
    ],
    'Cut_Chip': [
        'cut resistance', 'chip resistance', 'puncture resistance',
        'damage resistance', 'impact resistance', 'off-road durability'
    ],
    'NVH': [
        'noise', 'vibration', 'comfort', 'nvh',
        'quiet ride', 'sound absorbing', 'ride comfort'
    ],
    'Durability': [
        'load capacity', 'endurance', 'heavy duty', 'heavy-duty',
        'load index', 'strength', 'rupture', 'fatigue'
    ],
    'HP': [
        'high speed', 'speed rating', 'thermal stability',
        'centrifugal force', 'high velocity', 'speed durability'
    ],
    'Smart_Connected': [
        'sensor', 'tpms', 'tire pressure monitoring system',
        'rfid', 'iot', 'connected tire', 'smart tire',
        'telematics', 'data collection', 'wireless',
        'signal processing', 'transmission', 'receiver'
    ],
    'Sustainability': [
        'sustainable', 'recycled', 'bio-based', 'green tire',
        'environmentally friendly', 'low environmental impact',
        'renewable material', 'reclaimed rubber', 'bio-derived',
        'carbon footprint', 'circular economy'
    ],
}
# NOTE (taxonomy fix): 'Zero_Degree' and 'Electrical_Properties' used to live
# here under Performance Targets. Both describe *how* a tire is built
# (a belt architecture, a conductive-material property) rather than a
# performance outcome, so they were moved to TECHNOLOGY_LEVERS below as
# 'Zero_Degree_Belt' and merged into 'Electrical_Functionality'. This
# changes Performance_Targets/Technology_Levers label counts compared to
# reports generated before this taxonomy fix.


TECHNOLOGY_LEVERS = {
    'Compound': [
        'rubber composition', 'rubber compound', 'rubber component',
        'elastomer', 'synthetic rubber', 'natural rubber',
        'carbon black', 'silica', 'phr', 'filler', 'polybutadiene',
        'styrene butadiene', 'ssbr', 's-sbr', 'resin',
        'polymer', 'additive'
    ],
    'Tread_Design': [
        'tread pattern', 'tread portion', 'tire tread',
        'circumferential groove', 'lateral groove', 'main groove',
        'lug groove', 'land portion', 'block row', 'sipe', 'siping',
        'pitch sequence', 'shoulder land', 'pattern design'
    ],
    'Structure': [
        'carcass', 'belt', 'sidewall', 'bead', 'reinforcement',
        'ply', 'cord', 'breaker', 'liner', 'apex',
        'innerliner', 'chafer'
    ],
    'Process': [
        'vulcanization', 'curing', 'mixing', 'extrusion',
        'molding', 'shaping', 'assembly',
        'manufacturing method', 'production method'
    ],
    'Smart': [
        'sensor', 'tpms', 'tire pressure monitoring system',
        'rfid', 'iot', 'connected tire', 'smart tire',
        'telematics', 'data collection', 'wireless',
        'signal processing', 'transmission', 'receiver'
    ],

    # Merged from the old Performance-Targets 'Electrical_Properties' entry —
    # this is a material/structural property (how the tire is built), not a
    # performance outcome.
    'Electrical_Functionality': [
        'conductivity', 'conductive', 'electrically conductive',
        'electrical conductivity', 'electric conductivity',
        'conductive rubber', 'conductive rubber composition',
        'conductive layer', 'conductive member', 'conductive path',
        'conductive portion',
        'resistivity', 'electrical resistivity', 'electric resistivity',
        'volume resistivity', 'surface resistivity', 'specific resistance',
        'electrical resistance', 'electric resistance', 'resistance value',
        'volume resistance', 'surface resistance',
        'antistatic', 'anti-static', 'static electricity', 'electrostatic',
        'electrostatic discharge', 'esd',
        'ohm', 'ohms', 'omega', 'Ω', 'mohm', 'megaohm', 'mega-ohm',
        'mω', 'mΩ', 'kohm', 'kiloohm', 'kilo-ohm', 'kΩ',
        '10^6 ohm', '10^7 ohm', '10^8 ohm', '10^9 ohm',
        '10^10 ohm', '10^11 ohm', '10^12 ohm'
    ],

    # Moved from the old Performance-Targets 'Zero_Degree' entry — this
    # names a belt construction technique, not a performance outcome.
    'Zero_Degree_Belt': [
        'zero-degree', 'zero degree', 'low angle belt', 'low angle',
        'cap ply', 'circumferential belt', 'hoop belt',
        'hoop stress', 'helically wound'
    ]
}

print(f"✔ Segment keywords (master)     : {list(SEGMENT_KEYWORDS.keys())}")
print(f"✔ Segment safe phrases          : {list(SEGMENT_SAFE_PHRASES.keys())}")
print(f"✔ Segment risky single tokens   : {list(SEGMENT_RISKY_SINGLE_TOKENS.keys())}")
print(f"✔ Tire companies                : {len(TIRE_COMPANIES)}")
print(f"✔ Performance Targets           : {len(PERFORMANCE_TARGETS)}")
print(f"✔ Technology Levers             : {len(TECHNOLOGY_LEVERS)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: APPLY ALL CLASSIFICATIONS TO DATAFRAME
# Runs segment, assignee, taxonomy, AI/ML, and uniqueness scoring
# in correct dependency order. Adds all new columns to df.
# ═══════════════════════════════════════════════════════════════

# ── 1. Segment Classification ─────────────────────────────────
df['Segment'] = df.apply(
    lambda row: classify_segment(
        row['Abstract_lower'], row['Claims_lower'],
        row['Title_lower'], SEGMENT_SAFE_PHRASES, SEGMENT_RISKY_SINGLE_TOKENS
    ), axis=1
)

print(f"✔ Segment classification complete:\n{df['Segment'].value_counts().to_string()}\n")

# ── 2. Assignee Normalization ─────────────────────────────────
df['Assignee_Clean']      = df['Assignee'].apply(clean_assignee)
df['Assignee_Normalized'] = df.apply(
    lambda row: match_company(row['Assignee_Clean'], TIRE_COMPANIES, raw_text=row['Assignee']),
    axis=1
)
print(f"✔ Assignee normalization complete:\n{df['Assignee_Normalized'].value_counts().to_string()}\n")

# ── 3. Performance Targets ────────────────────────────────────
df['Performance_Targets'] = df.apply(
    lambda row: apply_categorization(
        row['Abstract_lower'], row['Claims_lower'], PERFORMANCE_TARGETS
    ), axis=1
)
df['PT_Evidence'] = df.apply(
    lambda row: apply_evidence(
        row['Abstract_lower'], row['Claims_lower'], PERFORMANCE_TARGETS
    ), axis=1
)

# ── 4. Technology Levers ──────────────────────────────────────
df['Technology_Levers'] = df.apply(
    lambda row: apply_categorization(
        row['Abstract_lower'], row['Claims_lower'], TECHNOLOGY_LEVERS
    ), axis=1
)
df['TL_Evidence'] = df.apply(
    lambda row: apply_evidence(
        row['Abstract_lower'], row['Claims_lower'], TECHNOLOGY_LEVERS
    ), axis=1
)
print("✔ Taxonomy categorization complete.")

# ── 5. AI/ML Detection ────────────────────────────────────────
df['AI_ML_Flag']     = (
    df['Title_lower'] + ' ' + df['Abstract_lower'] + ' ' + df['Claims_lower']
).apply(detect_aiml)
df['Contains_AI_ML'] = df['AI_ML_Flag'] != 'None'
print(f"✔ AI/ML patents detected: {df['Contains_AI_ML'].sum()}")

# ── 6. Count Scores ───────────────────────────────────────────
df['TL_Count'] = df['Technology_Levers'].apply(
    lambda x: len([i for i in str(x).split('|')
                   if i.strip() not in ('None', '')])
)
df['PT_Count'] = df['Performance_Targets'].apply(
    lambda x: len([i for i in str(x).split('|')
                   if i.strip() not in ('None', '')])
)

# ── 7. Uniqueness Score ───────────────────────────────────────
# TL_Count + PT_Count + AI/ML bonus (x2) + Sustainability bonus
df['Uniqueness_Score'] = (
    df['TL_Count'] +
    df['PT_Count'] +
    df['Contains_AI_ML'].astype(int) * 2 +
    df['Performance_Targets'].str.contains('Sustainability', na=False).astype(int)
)
print(f"✔ Uniqueness scoring complete. Max score: {df['Uniqueness_Score'].max()}")

# ── 8. Save final classified file ─────────────────────────────
OUTPUT_PATH = os.path.join(FILE_DIR, f"Patents_{MONTH_NAME.replace(' ', '_')}_classified.xlsx")
df.to_excel(OUTPUT_PATH, index=False)
print(f"\n✔ Classified file saved:\n  {OUTPUT_PATH}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6D: PATENT FAMILY DEDUPLICATION
# One invention can be published multiple times (US, EP, WO, CN...) under
# the same Simple Family ID. From here on, all analytics (charts, scoring,
# notable-patent tables, the Word report) run on one representative record
# per family, so a single invention is not counted 2-3x. The full
# publication-level data stays available as `all_publications_df` and was
# already saved in full to the classified Excel export above.
# ═══════════════════════════════════════════════════════════════

FAMILY_ID_COL = 'Simple Family ID'

all_publications_df = df.copy()

if FAMILY_ID_COL not in df.columns:
    print(f"⚠ '{FAMILY_ID_COL}' column not found — skipping family dedup; "
          f"treating every record as its own family.")
    df['_Family_Key'] = df['Record Number'].astype(str)
else:
    fam_id = df[FAMILY_ID_COL].astype(str).str.strip()
    blank_family = fam_id.str.lower().isin(['', 'nan', 'none', '0'])
    # Records without a usable family id are treated as singleton families
    # keyed on their own Record Number, so they are never silently merged
    # with an unrelated record just because both have a blank family id.
    df['_Family_Key'] = np.where(blank_family, 'REC_' + df['Record Number'].astype(str), fam_id)


def _pick_family_representative(group):
    """Prefer a WO/PCT publication, else the earliest publication date."""
    wo_rows = group[group['Publication Country'].astype(str).str.upper() == 'WO']
    pool = wo_rows if len(wo_rows) else group
    pool = pool.sort_values('Publication/Issue Date', na_position='last')
    return pool.iloc[0]


family_records = []
for key, group in df.groupby('_Family_Key', sort=False):
    rep = _pick_family_representative(group)
    members = sorted(group['Record Number'].astype(str).unique())
    countries = sorted(group['Publication Country'].astype(str).unique())
    family_records.append({
        '_Family_Key': key,
        'Representative_Record_Number': rep['Record Number'],
        'Family_Member_Records': ' | '.join(members),
        'Family_Member_Countries': ' | '.join(countries),
        'Family_Member_Count': len(members),
    })

family_lookup = pd.DataFrame(family_records)

df = df.merge(family_lookup, on='_Family_Key', how='left')
df['Is_Family_Representative'] = (
    df['Record Number'].astype(str) == df['Representative_Record_Number'].astype(str)
)

TOTAL_PUBLICATIONS = len(df)
TOTAL_UNIQUE_FAMILIES = df['_Family_Key'].nunique()
multi_member_families = int((family_lookup['Family_Member_Count'] > 1).sum())

print("✔ Family deduplication complete.")
print(f"  Publication records processed  : {TOTAL_PUBLICATIONS}")
print(f"  Unique simple families          : {TOTAL_UNIQUE_FAMILIES}")
if 'Extended Family ID' in df.columns:
    print(f"  Unique extended families        : {df['Extended Family ID'].astype(str).nunique()}")
print(f"  Families with >1 publication here: {multi_member_families}")

# Everything from here on (charts, scoring, notable-patent tables, the Word
# report) operates on one row per invention.
family_df = df[df['Is_Family_Representative']].copy().reset_index(drop=True)
print(f"  Rows feeding downstream analytics: {len(family_df)}")

df = family_df


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6E: CPC-BASED CROSS-VALIDATION (TECHNOLOGY LEVERS + SUSTAINABILITY)
# Uses the CPC codes already present in the PatSeer export (previously
# unused) as an independent, code-based signal to sanity-check the
# keyword-based classification. This does NOT overwrite Technology_Levers
# / Performance_Targets — it adds separate CPC_* columns and prints
# agreement/disagreement stats, so classification quality is measurable
# rather than assumed. Group mappings below are the stable, publicly
# documented CPC scheme definitions for the B60C/B29D tire classes (not
# derived from this month's data); Y02T is CPC's own dedicated
# climate-change-mitigation-in-transport tagging scheme, so its presence
# is a direct, code-based sustainability signal independent of keywords.
# ═══════════════════════════════════════════════════════════════

CPC_TECH_LEVER_MAP = {
    'Compound':     ['B60C1', 'C08K', 'C08L', 'C08F', 'C08C', 'C08G', 'C08J'],
    'Tread_Design': ['B60C11', 'B60C2011'],
    'Structure':    ['B60C9', 'B60C2009', 'B60C13', 'B60C15', 'D07B', 'B32B'],
    'Process':      ['B29D', 'B29C', 'B60C25'],
    'Smart':        ['B60C23', 'G06N', 'G06T', 'G06K', 'G07C', 'H04W', 'H04L'],
}
CPC_SUSTAINABILITY_PREFIX = 'Y02T'


def _cpc_groups(cpc_cell):
    """Extract normalized CPC group prefixes (e.g. 'B60C1', 'C08K') from a
    semicolon-separated CPC cell."""
    if pd.isna(cpc_cell):
        return set()
    groups = set()
    for code in str(cpc_cell).split(';'):
        code = code.strip()
        m = re.match(r'^([A-Z]\d{2}[A-Z]\d*)', code)
        if m:
            groups.add(m.group(1))
    return groups


def cpc_technology_levers(cpc_cell):
    groups = _cpc_groups(cpc_cell)
    matched = [
        lever for lever, prefixes in CPC_TECH_LEVER_MAP.items()
        if any(any(g.startswith(p) for p in prefixes) for g in groups)
    ]
    return ' | '.join(matched) if matched else 'None'


def cpc_sustainability_flag(cpc_cell):
    groups = _cpc_groups(cpc_cell)
    return any(g.startswith(CPC_SUSTAINABILITY_PREFIX) for g in groups)


if 'CPC' in df.columns:
    df['CPC_Technology_Levers'] = df['CPC'].apply(cpc_technology_levers)
    df['CPC_Sustainability_Flag'] = df['CPC'].apply(cpc_sustainability_flag)

    def _labels(cell):
        return set(x.strip() for x in str(cell).split('|') if x.strip() not in ('', 'None'))

    has_cpc_signal = df['CPC_Technology_Levers'] != 'None'
    agree = df.apply(
        lambda r: bool(_labels(r['Technology_Levers']) & _labels(r['CPC_Technology_Levers'])),
        axis=1
    )
    n_signal = int(has_cpc_signal.sum())
    n_agree = int((agree & has_cpc_signal).sum())

    print("✔ CPC cross-validation complete.")
    print(f"  Records with a CPC-derived technology-lever signal: {n_signal}/{len(df)}")
    if n_signal:
        print(f"  Keyword vs. CPC agreement on at least one lever: {n_agree}/{n_signal} "
              f"({100*n_agree/n_signal:.0f}%)")

    disagreement = df[has_cpc_signal & ~agree]
    if len(disagreement):
        print(f"  ⚠ {len(disagreement)} record(s) where CPC suggests a technology lever "
              f"the keyword classifier missed entirely — see CPC_Technology_Levers for "
              f"review candidates.")

    cpc_sustain_only = df[
        df['CPC_Sustainability_Flag'] &
        ~df['Performance_Targets'].str.contains('Sustainability', na=False)
    ]
    if len(cpc_sustain_only):
        print(f"  ⚠ {len(cpc_sustain_only)} record(s) carry the CPC Y02T sustainability code "
              f"but were NOT keyword-tagged Sustainability — review candidates.")

    # ── Close the gap: merge CPC-only signal into the primary columns ──
    # Text evidence stays as the primary signal; a CPC code only ADDS a
    # label the keywords missed, it never removes a keyword-based label.
    # TL_Evidence/PT_Evidence get an explicit "CPC code match: ..." note
    # so every added label stays traceable to why it's there — nothing is
    # silently blended in. TL_Count/PT_Count/Uniqueness_Score are
    # recomputed afterward so scoring reflects the merged labels.

    def _append_evidence(evidence, note):
        if not note:
            return evidence
        evidence = '' if str(evidence).strip() in ('', 'None', 'nan') else str(evidence).strip()
        return f"{evidence} | {note}" if evidence else note

    # Technology Levers: add any CPC-only lever to the end of the existing
    # (keyword-ordered) label list, without disturbing records that need
    # no change.
    df['TL_CPC_Added'] = df.apply(
        lambda r: ' | '.join(sorted(_labels(r['CPC_Technology_Levers']) - _labels(r['Technology_Levers']))),
        axis=1
    )
    n_tl_enriched = int((df['TL_CPC_Added'] != '').sum())

    def _merge_tl(existing, added_str):
        if not added_str:
            return existing
        existing_labels = [x.strip() for x in str(existing).split('|') if x.strip() not in ('', 'None')]
        for label in added_str.split(' | '):
            if label not in existing_labels:
                existing_labels.append(label)
        return ' | '.join(existing_labels) if existing_labels else 'None'

    df['Technology_Levers'] = df.apply(lambda r: _merge_tl(r['Technology_Levers'], r['TL_CPC_Added']), axis=1)
    df['TL_Evidence'] = df.apply(
        lambda r: _append_evidence(
            r['TL_Evidence'],
            f"CPC code match: {r['TL_CPC_Added']}" if r['TL_CPC_Added'] else ''
        ),
        axis=1
    )

    # Performance Targets: add Sustainability when CPC's own Y02T code
    # says so and keywords missed it.
    df['PT_Sustainability_CPC_Added'] = (
        df['CPC_Sustainability_Flag'] &
        ~df['Performance_Targets'].apply(lambda x: 'Sustainability' in _labels(x))
    )
    n_pt_enriched = int(df['PT_Sustainability_CPC_Added'].sum())

    def _add_sustainability(pt_value, add_flag):
        if not add_flag:
            return pt_value
        labels = [x.strip() for x in str(pt_value).split('|') if x.strip() not in ('', 'None')]
        if 'Sustainability' not in labels:
            labels.append('Sustainability')
        return ' | '.join(labels) if labels else 'None'

    df['Performance_Targets'] = df.apply(
        lambda r: _add_sustainability(r['Performance_Targets'], r['PT_Sustainability_CPC_Added']), axis=1
    )
    df['PT_Evidence'] = df.apply(
        lambda r: _append_evidence(
            r['PT_Evidence'],
            'CPC code match: Sustainability (Y02T)' if r['PT_Sustainability_CPC_Added'] else ''
        ),
        axis=1
    )

    # Recompute everything that depends on Technology_Levers/Performance_Targets
    # counts, so the merge actually flows through to scoring and the report.
    df['TL_Count'] = df['Technology_Levers'].apply(
        lambda x: len([i for i in str(x).split('|') if i.strip() not in ('None', '')])
    )
    df['PT_Count'] = df['Performance_Targets'].apply(
        lambda x: len([i for i in str(x).split('|') if i.strip() not in ('None', '')])
    )
    df['Uniqueness_Score'] = (
        df['TL_Count'] +
        df['PT_Count'] +
        df['Contains_AI_ML'].astype(int) * 2 +
        df['Performance_Targets'].str.contains('Sustainability', na=False).astype(int)
    )

    print(f"\n✔ CPC gap closed — merged into Technology_Levers / Performance_Targets:")
    print(f"  Technology_Levers enriched by a CPC-only lever : {n_tl_enriched} record(s)")
    print(f"  Performance_Targets enriched with Sustainability (CPC Y02T): {n_pt_enriched} record(s)")
    print(f"  TL_Count / PT_Count / Uniqueness_Score recomputed to reflect the merge.")
else:
    print("⚠ 'CPC' column not found — skipping CPC cross-validation.")


In [ ]:

# ═══════════════════════════════════════════════════════════════
# CELL 6B: PHASE-1 INSIGHTS
# Adds:
#   1) Tire region classification
#   2) TL/PT breadth metrics
#   3) TL-PT novelty index
#   4) Top patent interest score + Top 5 patents of the month
# ═══════════════════════════════════════════════════════════════

# ── Helper utilities ──────────────────────────────────────────
def split_labels(value):
    return [x.strip() for x in str(value).split('|') if x.strip() not in ('', 'None', 'nan')]

def safe_minmax(series):
    s = pd.to_numeric(series, errors='coerce').fillna(0)
    if len(s) == 0:
        return s
    smin, smax = s.min(), s.max()
    if smax == smin:
        return pd.Series(np.ones(len(s)), index=s.index) if smax > 0 else pd.Series(np.zeros(len(s)), index=s.index)
    return (s - smin) / (smax - smin)

# ── 1. Tire Region Classification ─────────────────────────────
TIRE_REGION_KEYWORDS = {
    'Tread': [
        'tread', 'tread pattern', 'tread rubber', 'tread block', 'cap tread',
        'crown pattern', 'groove', 'sipe', 'lug groove', 'rib groove'
    ],
    'Shoulder': [
        'shoulder', 'shoulder rib', 'shoulder groove', 'edge rib', 'tread edge'
    ],
    'Belt': [
        'belt', 'belt layer', 'belt ply', 'breaker', 'breaker ply',
        'steel cord belt', 'belt cord', 'cap ply', 'band ply', 'reinforcing belt'
    ],
    'Carcass': [
        'carcass', 'carcass ply', 'body ply', 'radial ply', 'cord layer', 'ply cord'
    ],
    'Sidewall': [
        'sidewall', 'side wall', 'side portion', 'rim cushion', 'flex cracking'
    ],
    'Bead': [
        'bead', 'bead core', 'bead filler', 'chafer', 'apex', 'bead portion'
    ],
    'Inner_Liner': [
        'inner liner', 'innerliner', 'airtight layer', 'gas barrier layer', 'halobutyl liner'
    ]
}

def classify_tire_regions(title, abstract, claims, region_dict):
    text = f"{str(title)} {str(title)} {str(abstract)} {str(claims)}".lower()
    matched = []
    for region, keywords in region_dict.items():
        if any(kw in text for kw in keywords):
            matched.append(region)
    return ' | '.join(matched) if matched else 'General'

df['Tire_Region'] = df.apply(
    lambda row: classify_tire_regions(
        row.get('Title_lower', ''),
        row.get('Abstract_lower', ''),
        row.get('Claims_lower', ''),
        TIRE_REGION_KEYWORDS
    ),
    axis=1
)
df['Region_Count'] = df['Tire_Region'].apply(lambda x: len(split_labels(x)) if str(x) != 'General' else 0)

print("✔ Tire region classification complete:")
print(df['Tire_Region'].value_counts().head(10).to_string())
print()

# ── 2. Breadth / Complexity Metrics ───────────────────────────
df['Total_Label_Breadth'] = df['TL_Count'] + df['PT_Count']
df['Is_Multi_TL'] = df['TL_Count'] >= 2
df['Is_Multi_PT'] = df['PT_Count'] >= 2
df['Is_Cross_Functional'] = (df['TL_Count'] >= 2) & (df['PT_Count'] >= 2)

# Simple categorical tag for quick management reporting
def breadth_band(total_labels):
    if total_labels >= 6:
        return 'Very High'
    elif total_labels >= 4:
        return 'High'
    elif total_labels >= 2:
        return 'Medium'
    return 'Low'

df['Breadth_Band'] = df['Total_Label_Breadth'].apply(breadth_band)

# ── 3. Novelty Index based on rare TL-PT combinations ─────────
pair_counter = Counter()

for _, row in df.iterrows():
    tls = split_labels(row.get('Technology_Levers', ''))
    pts = split_labels(row.get('Performance_Targets', ''))
    pairs = {(tl, pt) for tl in tls for pt in pts}
    pair_counter.update(pairs)

def compute_pair_novelty(row, pair_counter):
    tls = split_labels(row.get('Technology_Levers', ''))
    pts = split_labels(row.get('Performance_Targets', ''))
    pairs = {(tl, pt) for tl in tls for pt in pts}
    if not pairs:
        return 0.0
    rarity_scores = [1.0 / pair_counter[pair] for pair in pairs if pair_counter[pair] > 0]
    return float(np.mean(rarity_scores)) if rarity_scores else 0.0

df['Novelty_Index_Raw'] = df.apply(lambda row: compute_pair_novelty(row, pair_counter), axis=1)
df['Novelty_Index'] = (safe_minmax(df['Novelty_Index_Raw']) * 100).round(1)

# ── 4. Patent Interest Score + Top 5 Patents ──────────────────
# Transparent weighted score:
#   45% novelty
#   20% technology breadth
#   20% performance breadth
#   10% region specificity
#    5% cross-functional bonus
df['TL_Breadth_Norm'] = (safe_minmax(df['TL_Count']) * 100).round(1)
df['PT_Breadth_Norm'] = (safe_minmax(df['PT_Count']) * 100).round(1)
df['Region_Breadth_Norm'] = (safe_minmax(df['Region_Count']) * 100).round(1)
df['Cross_Functional_Bonus'] = np.where(df['Is_Cross_Functional'], 100, 0)

df['Patent_Interest_Score'] = (
    0.45 * df['Novelty_Index'] +
    0.20 * df['TL_Breadth_Norm'] +
    0.20 * df['PT_Breadth_Norm'] +
    0.10 * df['Region_Breadth_Norm'] +
    0.05 * df['Cross_Functional_Bonus']
).round(1)

def build_interest_reason(row):
    reasons = []
    if row.get('Novelty_Index', 0) >= 70:
        reasons.append('rare TL-PT combination')
    if row.get('TL_Count', 0) >= 2:
        reasons.append(f"{int(row['TL_Count'])} technology levers")
    if row.get('PT_Count', 0) >= 2:
        reasons.append(f"{int(row['PT_Count'])} performance targets")
    if row.get('Region_Count', 0) >= 1:
        reasons.append(str(row.get('Tire_Region', 'General')).replace(' | ', ', '))
    return '; '.join(reasons[:4]) if reasons else 'focused single-theme patent'

df['Interest_Reason'] = df.apply(build_interest_reason, axis=1)

top5_patents = (
    df.sort_values(
        ['Patent_Interest_Score', 'Novelty_Index', 'Publication/Issue Date'],
        ascending=[False, False, False]
    )
    .loc[:, [
        'Record Number', 'Title', 'Assignee_Normalized', 'Publication/Issue Date',
        'Technology_Levers', 'Performance_Targets', 'Tire_Region',
        'TL_Count', 'PT_Count', 'Novelty_Index', 'Patent_Interest_Score',
        'Interest_Reason'
    ]]
    .head(5)
    .reset_index(drop=True)
)

print("✔ Phase-1 scoring complete.")
print(f"  Avg Novelty Index      : {df['Novelty_Index'].mean():.1f}")
print(f"  Avg Patent Interest    : {df['Patent_Interest_Score'].mean():.1f}")
print(f"  Multi-TL patents       : {int(df['Is_Multi_TL'].sum())}")
print(f"  Multi-PT patents       : {int(df['Is_Multi_PT'].sum())}")
print(f"  Cross-functional patents: {int(df['Is_Cross_Functional'].sum())}")
print("\nTop 5 patents of the month:")
display(top5_patents[['Record Number', 'Assignee_Normalized', 'Patent_Interest_Score', 'Novelty_Index', 'Interest_Reason']])

# Save enhanced classified file
OUTPUT_PATH_PHASE1 = os.path.join(FILE_DIR, f"Patents_{MONTH_NAME.replace(' ', '_')}_classified_phase1.xlsx")
df.to_excel(OUTPUT_PATH_PHASE1, index=False)
top5_patents.to_excel(os.path.join(FILE_DIR, f"Top5_Patents_{MONTH_NAME.replace(' ', '_')}.xlsx"), index=False)

print(f"\n✔ Enhanced file saved:\n  {OUTPUT_PATH_PHASE1}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6F: NOTABLE-PATENT SET FOR PDF DRAWING EXTRACTION
# At low volume (the original ~84/month query) it was feasible to bulk
# download every patent's PDF and embed drawing pages for the full list.
# At the current query's volume (500+/month) that stopped being
# realistic. From here on, PDF drawing extraction/embedding in the Word
# report is scoped to the same "notable" patents already highlighted in
# Section 4 (Top Interest Score, Top Novelty, Cross-domain, AI/ML,
# Sustainability, Smart/Connected) — every patent still gets a full text
# card either way. This cell computes that set once, up front, and saves
# the record-number list so PDFs can be bulk-downloaded for just this
# subset instead of the full monthly volume.
# ═══════════════════════════════════════════════════════════════

_top5_interest_records = set(top5_patents['Record Number'].astype(str))

_top5_novelty_records = set(
    df.sort_values(['Novelty_Index', 'Patent_Interest_Score'], ascending=False)
      .head(5)['Record Number'].astype(str)
)

_cross_domain_records = set(df[df['PT_Count'] >= 3]['Record Number'].astype(str))
_aiml_records = set(df[df['Contains_AI_ML']]['Record Number'].astype(str))

# NOTE: Sustainability and Smart/Connected were tried here originally but
# dropped from this trigger after checking real numbers — Smart/Connected
# alone matched 157/564 (28%) records on the June 2026 export (generic
# terms like 'sensor'/'wireless' plus the CPC-merge from Cell 6E), which
# defeats the purpose of narrowing PDF handling to a manageable set. Both
# categories are still fully visible as text tables in Section 4 — they
# just don't trigger PDF drawing extraction on their own.
_notable_candidates = (
    _top5_interest_records | _top5_novelty_records | _cross_domain_records | _aiml_records
)

# ── Hard cap (MAX_NOTABLE_PATENTS, Cell 1) ────────────────────
# The category union above is *emergent* — its size floats with the data
# and can exceed what one person can bulk-download in a month. So it is
# treated only as the candidate pool: the Top-5 Interest and Top-5
# Novelty sets are always kept, and remaining slots are filled from the
# other candidates in Patent Interest Score order until the cap.
_must_keep = _top5_interest_records | _top5_novelty_records
_fill_pool = (
    df[df['Record Number'].astype(str).isin(_notable_candidates - _must_keep)]
    .sort_values(['Patent_Interest_Score', 'Novelty_Index'], ascending=False)
    ['Record Number'].astype(str).tolist()
)
_n_fill = max(MAX_NOTABLE_PATENTS - len(_must_keep), 0)
NOTABLE_RECORD_NUMBERS = _must_keep | set(_fill_pool[:_n_fill])

df['Is_Notable_Patent'] = df['Record Number'].astype(str).isin(NOTABLE_RECORD_NUMBERS)

print(f"✔ Notable-patent set for PDF drawings: {len(NOTABLE_RECORD_NUMBERS)}/{len(df)} records "
      f"(cap: {MAX_NOTABLE_PATENTS})")
print(f"  Candidate pool before cap: {len(_notable_candidates)}")
print(f"    Top Interest Score  : {len(_top5_interest_records)} (always kept)")
print(f"    Top Novelty         : {len(_top5_novelty_records)} (always kept)")
print(f"    Cross-domain (>=3 PT): {len(_cross_domain_records)}")
print(f"    AI/ML               : {len(_aiml_records)}")
if len(_notable_candidates) > MAX_NOTABLE_PATENTS:
    print(f"  {len(_notable_candidates) - len(NOTABLE_RECORD_NUMBERS)} candidate(s) dropped by the cap "
          f"(lowest Interest Score first) — raise MAX_NOTABLE_PATENTS in Cell 1 if you want more.")

notable_pdf_list_path = os.path.join(FILE_DIR, f"Notable_Patents_PDFs_Needed_{MONTH_NAME.replace(' ', '_')}.xlsx")
df[df['Is_Notable_Patent']][['Record Number', 'Title', 'Assignee_Normalized', 'PDF Link']] \
    .to_excel(notable_pdf_list_path, index=False)
print(f"\n✔ Notable-patent list saved (bulk-download PDFs for just this set):\n  {notable_pdf_list_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6C: USE EXISTING PATSEER AI SUMMARY
# Uses "Problem Being Solved (AI Sum.)" as the summary source.
# Falls back to Abstract only if the PatSeer AI field is empty.
# ═══════════════════════════════════════════════════════════════

print("─── Preparing Summary Column ─────────────────────────────")

summary_source_col = "Problem Being Solved (AI Sum.)"
fallback_col = "Abstract"

if summary_source_col not in df.columns:
    raise ValueError(
        f"Column '{summary_source_col}' not found. "
        f"Available columns: {list(df.columns)}"
    )

def clean_summary_text(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\\n", " ").replace("\n", " ").strip()
    x = " ".join(x.split())
    return x

def make_summary(row):
    problem_text = clean_summary_text(row.get(summary_source_col, ""))
    
    if problem_text:
        return problem_text[:700]
    
    # fallback only if Problem Being Solved is empty
    abstract_text = clean_summary_text(row.get(fallback_col, ""))
    if abstract_text:
        return abstract_text[:700]
    
    return "No summary available."

df["AI_Summary"] = df.apply(make_summary, axis=1)

valid = (df["AI_Summary"] != "No summary available.").sum()

print(f"✔ AI_Summary column ready: {valid}/{len(df)} patents populated.")
print(f"  Primary source: {summary_source_col}")
print(f"  Fallback source: {fallback_col}")

print("\n── Sample Summaries ──────────────────────────────────────")
for i in range(min(3, len(df))):
    rec = df.iloc[i].get("Record Number", f"Row {i+1}")
    print(f"\n📌 Record {rec}:")
    print(f"   {df.iloc[i]['AI_Summary'][:250]}...")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: CHART 1 — Patent Count by Publication Country
# Bar chart (left) + Pie chart (right) side by side.
# Countries below 2% threshold grouped into 'Others' on pie.
# ═══════════════════════════════════════════════════════════════

country_counts = (
    df['Publication Country'].value_counts()
    .reset_index()
)
country_counts.columns = ['Country', 'Patent Count']

# Group small countries into 'Others' for pie only
threshold = 0.02 * country_counts['Patent Count'].sum()
main  = country_counts[country_counts['Patent Count'] >= threshold]
small = country_counts[country_counts['Patent Count'] <  threshold]
if len(small) > 0:
    pie_data = pd.concat([
        main,
        pd.DataFrame([['Others', small['Patent Count'].sum()]],
                     columns=['Country', 'Patent Count'])
    ], ignore_index=True)
else:
    pie_data = main.copy()

n_bar      = len(country_counts)
n_pie      = len(pie_data)
bar_colors = [plt.cm.viridis(i / n_bar) for i in range(n_bar)]
pie_colors = [plt.cm.viridis(i / n_pie) for i in range(n_pie)]

fig, (ax_bar, ax_pie) = plt.subplots(
    1, 2, figsize=(18, 7),
    gridspec_kw={'width_ratios': [1.4, 1]}
)
fig.suptitle('Patent Count by Publication Country',
             fontsize=15, fontweight='bold', y=1.01)

# Left: Bar chart
x_pos = list(range(n_bar))
bars  = ax_bar.bar(x_pos, country_counts['Patent Count'],
                   color=bar_colors, edgecolor='white', linewidth=0.5)
for bar in bars:
    ax_bar.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5, str(int(bar.get_height())),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_bar.xaxis.set_major_locator(FixedLocator(x_pos))
ax_bar.set_xticklabels(country_counts['Country'], rotation=45,
                       ha='right', fontsize=13)
ax_bar.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax_bar.tick_params(axis='y', labelsize=13)
ax_bar.set_xlabel('Publication Country', fontsize=13)
ax_bar.set_ylabel('Number of Patents', fontsize=13)
ax_bar.set_title('Distribution (Count)', fontsize=13, pad=10)
ax_bar.grid(axis='y', linestyle='--', alpha=0.5)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

# Right: Pie chart
wedges, texts, autotexts = ax_pie.pie(
    pie_data['Patent Count'], labels=pie_data['Country'],
    autopct='%1.1f%%', startangle=140, colors=pie_colors,
    wedgeprops=dict(edgecolor='white', linewidth=1.5), pctdistance=0.82
)
for t in texts:     t.set_fontsize(13)
for a in autotexts: a.set_fontsize(13); a.set_fontweight('bold'); a.set_color('white')
ax_pie.set_title('Share (%)', fontsize=13, pad=10)

plt.tight_layout()
chart_path = os.path.join(FILE_DIR, 'chart_01_country_patents.png')
plt.savefig(chart_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✔ Chart saved: {chart_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: CHART 2 — Top 5 Assignees by Patent Count
# Shows Top 5 known assignees + 'Others' (unmatched entities).
# ═══════════════════════════════════════════════════════════════

# ── Prepare Data ──────────────────────────────────────────────
assignee_counts = df['Assignee_Normalized'].value_counts().reset_index()
assignee_counts.columns = ['Company', 'Patent Count']

# Separate top 5 known companies from Others
_pseudo = ['Others', 'Other Tire/Rubber Manufacturer']
top5 = assignee_counts[~assignee_counts['Company'].isin(_pseudo)].head(5).copy()
others_count = assignee_counts[assignee_counts['Company'].isin(_pseudo)]['Patent Count'].sum()

# Append Others as the last bar
others_row = pd.DataFrame([['Others', others_count]], columns=['Company', 'Patent Count'])
plot_data  = pd.concat([top5, others_row], ignore_index=True)

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
x_pos   = list(range(len(plot_data)))

# Others bar gets a distinct grey color; top 5 use viridis palette
colors = [plt.cm.viridis(i / len(top5)) for i in range(len(top5))] + ['#AAAAAA']

bars = ax.bar(x_pos, plot_data['Patent Count'], color=colors,
              edgecolor='white', linewidth=0.5)

# Data labels on top of each bar
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3, str(int(bar.get_height())),
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# ── Axis Formatting ───────────────────────────────────────────
ax.xaxis.set_major_locator(FixedLocator(x_pos))
ax.set_xticklabels(plot_data['Company'], rotation=0, fontsize=11)
ax.set_xlabel('Assignee', fontsize=12)
ax.set_ylabel('Number of Patents', fontsize=12)
ax.set_title('Top 5 Assignees by Patent Count (+ Others)',
             fontsize=14, fontweight='bold', pad=15)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Others annotation below the bar ──────────────────────────
ax.text(x_pos[-1], -1.8,
        f"({others_count} unmatched\nentities)",
        ha='center', va='top', fontsize=9,
        color='grey', fontstyle='italic')

plt.tight_layout()

# ── Save & Show ───────────────────────────────────────────────
chart_path = os.path.join(FILE_DIR, 'chart_03_top5_assignees.png')
plt.savefig(chart_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✔ Chart saved: {chart_path}")
print(f"  Top 5 assignees + Others ({others_count} patents)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: CHART 3 — Technology Levers & Performance Targets
# Two bar charts side by side. Explodes pipe-separated labels
# and counts each one individually. Excludes 'None' entries.
# ═══════════════════════════════════════════════════════════════

tl_counts = count_labels(df['Technology_Levers'])
pt_counts = count_labels(df['Performance_Targets'])

fig, (ax_tl, ax_pt) = plt.subplots(
    1, 2, figsize=(18, 7),
    gridspec_kw={'wspace': 0.35}
)
fig.suptitle('Patent Distribution: Technology Levers & Performance Targets',
             fontsize=15, fontweight='bold', y=1.02)

for ax, counts, title, xlabel in [
    (ax_tl, tl_counts, 'Technology Levers', 'Technology Lever'),
    (ax_pt, pt_counts, 'Performance Targets', 'Performance Target')
]:
    x_pos  = list(range(len(counts)))
    colors = [plt.cm.viridis(i / len(counts)) for i in range(len(counts))]
    bars   = ax.bar(x_pos, counts['Count'], color=colors,
                    edgecolor='white', linewidth=0.5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3, str(int(bar.get_height())),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.xaxis.set_major_locator(FixedLocator(x_pos))
    ax.set_xticklabels(counts['Label'], rotation=45, ha='right', fontsize=13)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=13)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel('Number of Patents', fontsize=13)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
chart_path = os.path.join(FILE_DIR, 'chart_04_tl_pt_distribution.png')
plt.savefig(chart_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✔ Chart saved: {chart_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: CHART 4 — Technology Levers vs Performance Targets Heatmap
# Viridis version with automatic annotation coloring
# ═══════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as path_effects
from matplotlib.colors import LogNorm, Normalize

print("=" * 80)
print("CHART 4 — Technology Levers vs Performance Targets Heatmap")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Build co-occurrence matrix
# ---------------------------------------------------------------------------

all_pt = list(PERFORMANCE_TARGETS.keys())
all_tl = list(TECHNOLOGY_LEVERS.keys())

matrix = pd.DataFrame(0, index=all_tl, columns=all_pt)

for _, row in df.iterrows():
    pt_labels = [
        p.strip()
        for p in str(row.get("Performance_Targets", "")).split("|")
        if p.strip() not in ("None", "", "nan") and p.strip() in all_pt
    ]

    tl_labels = [
        t.strip()
        for t in str(row.get("Technology_Levers", "")).split("|")
        if t.strip() not in ("None", "", "nan") and t.strip() in all_tl
    ]

    for pt in pt_labels:
        for tl in tl_labels:
            matrix.loc[tl, pt] += 1

# ---------------------------------------------------------------------------
# 2. Clean matrix
# ---------------------------------------------------------------------------

matrix_plot = matrix.loc[
    matrix.sum(axis=1) > 0,
    matrix.sum(axis=0) > 0
].copy()

matrix_plot = matrix_plot.loc[
    matrix_plot.sum(axis=1).sort_values(ascending=False).index,
    matrix_plot.sum(axis=0).sort_values(ascending=False).index
]

if matrix_plot.empty:
    raise ValueError(
        "The TL × PT co-occurrence matrix is empty. "
        "Please check Technology_Levers and Performance_Targets columns."
    )

max_count = int(matrix_plot.values.max())
nonzero_values = matrix_plot.values[matrix_plot.values > 0]

# ---------------------------------------------------------------------------
# 3. Viridis color settings
# ---------------------------------------------------------------------------

heatmap_cmap = plt.colormaps["viridis"].copy()

zero_cell_color = "#241542"   # dark purple close to viridis low-end
figure_bg = "#f2f0f7"         # soft light lavender background
axis_bg = zero_cell_color

heatmap_cmap.set_bad(color=zero_cell_color)

# Decide whether to use log scale
use_log_scale = False

if len(nonzero_values) > 0:
    median_nonzero = np.median(nonzero_values)
    if max_count >= 5 * median_nonzero and max_count >= 10:
        use_log_scale = True

# Mask zero cells so they appear as blank dark cells
plot_data = matrix_plot.replace(0, np.nan)
mask = plot_data.isna()

if use_log_scale:
    norm = LogNorm(vmin=1, vmax=max_count)
    colorbar_label = "Patent Count, log-scaled"
else:
    norm = Normalize(vmin=1, vmax=max_count)
    colorbar_label = "Patent Count"

# ---------------------------------------------------------------------------
# 4. Plot heatmap without seaborn annotations
#    We add custom annotations manually for better readability.
# ---------------------------------------------------------------------------

fig_width = max(14, 1.1 * matrix_plot.shape[1])
fig_height = max(7, 0.65 * matrix_plot.shape[0])

fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor=figure_bg)
ax.set_facecolor(axis_bg)

sns.heatmap(
    plot_data,
    annot=False,
    fmt="",
    cmap=heatmap_cmap,
    norm=norm,
    mask=mask,
    linewidths=0.9,
    linecolor="#d8d4e8",
    ax=ax,
    cbar_kws={
        "label": colorbar_label,
        "shrink": 0.85,
        "pad": 0.02
    }
)

# ---------------------------------------------------------------------------
# 5. Add automatic annotation coloring
# ---------------------------------------------------------------------------

def relative_luminance(rgba):
    r, g, b = rgba[:3]
    return 0.299 * r + 0.587 * g + 0.114 * b

for i in range(matrix_plot.shape[0]):
    for j in range(matrix_plot.shape[1]):
        value = int(matrix_plot.iloc[i, j])

        if value == 0:
            continue

        rgba = heatmap_cmap(norm(value))
        lum = relative_luminance(rgba)

        # For darker cells use white text, for brighter cells use black text
        if lum > 0.53:
            txt_color = "black"
            outline_color = "white"
        else:
            txt_color = "white"
            outline_color = "black"

        txt = ax.text(
            j + 0.5,
            i + 0.5,
            str(value),
            ha="center",
            va="center",
            fontsize=14,
            fontweight="bold",
            color=txt_color
        )

        # Add outline for better readability
        txt.set_path_effects([
            path_effects.Stroke(linewidth=2.0, foreground=outline_color),
            path_effects.Normal()
        ])

# ---------------------------------------------------------------------------
# 6. Formatting
# ---------------------------------------------------------------------------

ax.set_title(
    "Technology Levers vs Performance Targets\n"
    "Patent Co-occurrence Count",
    fontsize=17,
    fontweight="bold",
    pad=18,
    color="black"
)

ax.set_xlabel(
    "Performance Targets",
    fontsize=15,
    fontweight="bold",
    labelpad=12
)

ax.set_ylabel(
    "Technology Levers",
    fontsize=15,
    fontweight="bold",
    labelpad=12
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=40,
    ha="right",
    fontsize=13
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=13
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#2b2440")
    spine.set_linewidth(1.3)

# Colorbar formatting
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=11)
cbar.set_label(colorbar_label, fontsize=12, fontweight="bold")
cbar.ax.set_facecolor(figure_bg)

plt.tight_layout()

# ---------------------------------------------------------------------------
# 7. Save
# ---------------------------------------------------------------------------

chart_path = os.path.join(FILE_DIR, "chart_02_heatmap_pt_tl_viridis_readable.png")
plt.savefig(chart_path, dpi=300, bbox_inches="tight", facecolor=figure_bg)
plt.show()

print(f"✔ Chart saved: {chart_path}")

print("\nMatrix summary:")
print("-" * 80)
print(f"Technology levers shown   : {matrix_plot.shape[0]}")
print(f"Performance targets shown : {matrix_plot.shape[1]}")
print(f"Maximum co-occurrence     : {max_count}")
print(f"Log scale used            : {use_log_scale}")

In [ ]:

# ═══════════════════════════════════════════════════════════════
# CELL 10B: PHASE-1 VISUALS
#   A) Tire region distribution
#   B) Top 10 patents by interest score
# ═══════════════════════════════════════════════════════════════

# ── A. Tire Region Distribution ───────────────────────────────
region_counts = count_labels(df['Tire_Region'])
if len(region_counts):
    fig, ax = plt.subplots(figsize=(11, 6))
    colors = [plt.cm.viridis(i / max(len(region_counts), 1)) for i in range(len(region_counts))]
    bars = ax.bar(region_counts['Label'], region_counts['Count'], color=colors, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                f"{int(bar.get_height())}", ha='center', va='bottom', fontsize=9)
    ax.set_title('Patent Distribution by Tire Region', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tire Region')
    ax.set_ylabel('Patent Count')
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    plt.savefig(os.path.join(FILE_DIR, 'chart_phase1_tire_region_distribution.png'), dpi=300, bbox_inches='tight')
    plt.show()

# ── B. Top 10 Patents by Patent Interest Score ────────────────
top10_plot = (
    df.sort_values(['Patent_Interest_Score', 'Novelty_Index'], ascending=False)
      .head(10)
      .copy()
)
top10_plot['Patent_Label'] = top10_plot['Record Number'].astype(str)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top10_plot['Patent_Label'][::-1], top10_plot['Patent_Interest_Score'][::-1])
for i, (_, row) in enumerate(top10_plot.iloc[::-1].iterrows()):
    ax.text(row['Patent_Interest_Score'] + 0.7, i,
            f"{row['Assignee_Normalized']} | N={row['Novelty_Index']:.1f}",
            va='center', fontsize=8)
ax.set_title('Top 10 Patents by Patent Interest Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Patent Interest Score')
ax.set_ylabel('Record Number')
plt.tight_layout()
plt.savefig(os.path.join(FILE_DIR, 'chart_phase1_top10_patents.png'), dpi=300, bbox_inches='tight')
plt.show()

# Quick management tables
region_summary = region_counts.rename(columns={'Label': 'Tire Region', 'Count': 'Patent Count'})
breadth_summary = (
    df['Breadth_Band'].value_counts()
      .rename_axis('Breadth Band')
      .reset_index(name='Patent Count')
)

print("Tire region summary:")
display(region_summary)

print("Breadth summary:")
display(breadth_summary)

print("Top 5 patents table:")
display(top5_patents)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: FINAL WORD REPORT GENERATOR
# ═══════════════════════════════════════════════════════════════

# ── Imports ───────────────────────────────────────────────────
try:
    from docx import Document
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'python-docx'], check=True)
    from docx import Document

import time
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# ── Dependency Check ──────────────────────────────────────────
required_cols = ['Assignee_Normalized', 'Technology_Levers',
                 'Performance_Targets', 'Segment',
                 'TL_Count', 'PT_Count', 'Contains_AI_ML', 'AI_ML_Flag',
                 'Tire_Region', 'Breadth_Band', 'Novelty_Index',
                 'Patent_Interest_Score', 'Interest_Reason']

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing columns: {missing}. Run Cells 3–6B first.")

if 'Condensed_Summary' not in df.columns:
    df['Condensed_Summary'] = ''
    print("⚠ Condensed_Summary not found — will generate summaries live.")
else:
    filled = df['Condensed_Summary'].astype(str).str.strip().replace('', pd.NA).notna().sum()
    print(f"✔ Condensed_Summary found — {filled}/{len(df)} patents have summaries.")

if 'AI_Summary' in df.columns:
    ai_filled = df['AI_Summary'].astype(str).str.strip().replace('', pd.NA).notna().sum()
    print(f"✔ AI_Summary found — {ai_filled}/{len(df)} patents have summaries.")
else:
    df['AI_Summary'] = ''
    print("⚠ AI_Summary not found — run Cell 6C first for smart summaries.")

# ── Build PDF link lookup dict once ───────────────────────────
from pathlib import Path
import glob
import os
import re
import pandas as pd

# Original PatSeer/export hyperlinks
if 'PDF_Link_URL' in df.columns:
    pdf_links = {
        str(row['Record Number']).strip(): str(row['PDF_Link_URL']).strip()
        for _, row in df.iterrows()
        if str(row.get('PDF_Link_URL', '')).strip().startswith('http')
    }
    print(f"✔ PatSeer/export PDF links loaded: {len(pdf_links)} valid URLs found.")
else:
    pdf_links = {}
    print("⚠ PDF_Link_URL column not found — PatSeer export links unavailable.")


# ═══════════════════════════════════════════════════════════════
# SAFE LOCAL PDF LINK HANDLING
# ═══════════════════════════════════════════════════════════════

def _safe_str(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ("nan", "none", ""):
        return ""
    return x


def _norm_pdf_key(value):
    value = _safe_str(value)
    if not value:
        return ""
    return re.sub(r'[^A-Za-z0-9]', '', value).upper()


def _patent_variants(value):
    key = _norm_pdf_key(value)
    variants = set()
    if not key:
        return []
    variants.add(key)
    m = re.match(r"^([A-Z]{2})(\d+)([A-Z]\d?)$", key)
    if m:
        country, number_body, kind = m.groups()
        variants.add(f"{country}{number_body.lstrip('0')}{kind}")
        ym = re.match(r"^(\d{4})(0+)(\d+)$", number_body)
        if ym:
            year, zeros, rest = ym.groups()
            variants.add(f"{country}{year}{rest}{kind}")
    return [v for v in variants if v]


def _path_to_file_uri(path_value):
    path_value = _safe_str(path_value)
    if not path_value:
        return ""
    try:
        p = Path(path_value)
        if not p.is_absolute():
            p = Path.cwd() / p
        p = p.resolve()
        if not p.exists():
            return ""
        return p.as_uri()
    except Exception as e:
        print(f"⚠ Could not create file URI for: {path_value} | {e}")
        return ""


def _build_local_pdf_links_auto():
    search_dirs = []
    candidate_dirs = [
        os.getcwd(),
        FILE_DIR if 'FILE_DIR' in globals() else None,
        os.path.dirname(FILE_PATH) if 'FILE_PATH' in globals() else None,
        os.path.join(os.getcwd(), "Downloaded_Patent_PDFs"),
        os.path.join(os.getcwd(), "PatSeer_PDFs"),
        os.path.join(FILE_DIR, "Downloaded_Patent_PDFs") if 'FILE_DIR' in globals() else None,
        os.path.join(FILE_DIR, "PatSeer_PDFs") if 'FILE_DIR' in globals() else None,
    ]
    for candidate in candidate_dirs:
        if candidate and os.path.isdir(candidate):
            candidate_abs = os.path.abspath(candidate)
            if candidate_abs not in search_dirs:
                search_dirs.append(candidate_abs)

    print("PDF search folders:")
    for d in search_dirs:
        print(" -", d)

    files = []
    for folder in search_dirs:
        files.extend(glob.glob(os.path.join(folder, "*.pdf")))
        files.extend(glob.glob(os.path.join(folder, "**", "*.pdf"), recursive=True))

    seen = set()
    unique_files = []
    for p in files:
        abs_p = os.path.abspath(p)
        if abs_p not in seen:
            seen.add(abs_p)
            unique_files.append(abs_p)
    files = sorted(unique_files)
    print(f"PDF files found: {len(files)}")

    pdf_index = []
    for p in files:
        stem = Path(p).stem
        pdf_index.append({
            "path":     os.path.abspath(p),
            "filename": os.path.basename(p),
            "key":      _norm_pdf_key(stem),
            "variants": _patent_variants(stem),
        })

    def find_pdf(record_no):
        record_no   = _safe_str(record_no)
        if not record_no:
            return "", "Missing"
        rec_key      = _norm_pdf_key(record_no)
        rec_variants = _patent_variants(record_no)
        if not rec_key and not rec_variants:
            return "", "Missing"

        # 1. Exact variant match
        matches = [item for item in pdf_index
                   if any(rv == pv for rv in rec_variants for pv in item["variants"])]
        matches = list({m["path"]: m for m in matches}.values())
        if len(matches) == 1:
            return matches[0]["path"], "Found - exact normalized variant"
        elif len(matches) > 1:
            return "", f"Ambiguous - exact variant matched {len(matches)} files"

        # 2. Full normalized key contained in filename
        matches = [item for item in pdf_index if rec_key and rec_key in item["key"]]
        matches = list({m["path"]: m for m in matches}.values())
        if len(matches) == 1:
            return matches[0]["path"], "Found - full key contained in filename"
        elif len(matches) > 1:
            return "", f"Ambiguous - full key matched {len(matches)} files"

        # 3. Variant appears in filename or filename in variant
        matches = []
        for item in pdf_index:
            for rv in rec_variants:
                if rv and (rv in item["key"] or item["key"] in rv):
                    matches.append(item)
                    break
        matches = list({m["path"]: m for m in matches}.values())
        if len(matches) == 1:
            return matches[0]["path"], "Found - contains normalized variant"
        elif len(matches) > 1:
            return "", f"Ambiguous - contains matched {len(matches)} files"

        # 4. Last 8/6/4 character fallback (unique only)
        best_key = max(rec_variants, key=len) if rec_variants else rec_key
        for n in [8, 6, 4]:
            if len(best_key) < n:
                continue
            suffix  = best_key[-n:]
            matches = [item for item in pdf_index
                       if suffix in item["key"] or any(suffix in pv for pv in item["variants"])]
            matches = list({m["path"]: m for m in matches}.values())
            if len(matches) == 1:
                return matches[0]["path"], f"Found - last{n} unique fallback"
            elif len(matches) > 1:
                return "", f"Ambiguous - last{n} matched {len(matches)} files"

        return "", "Missing"

    if 'Record Number' not in df.columns:
        print("⚠ Record Number column not found. Cannot map local PDFs.")
        return {}

    results = df['Record Number'].apply(find_pdf)
    df['Local_PDF_Path']          = results.apply(lambda x: x[0])
    df['PDF_Match_Status']        = results.apply(lambda x: x[1])
    df['Local_PDF_File']          = df['Local_PDF_Path'].apply(
        lambda x: os.path.basename(x) if _safe_str(x) else "")
    df['PDF_Status']              = df['Local_PDF_Path'].apply(
        lambda x: "Found" if _safe_str(x) and os.path.exists(_safe_str(x)) else "Missing")
    df['Local_PDF_Relative_Path'] = df['Local_PDF_Path'].apply(
        lambda x: os.path.relpath(x, os.getcwd())
        if _safe_str(x) and os.path.exists(_safe_str(x)) else "")
    df['Local_PDF_URI']           = df['Local_PDF_Path'].apply(_path_to_file_uri)

    local_links = {
        str(row['Record Number']).strip(): str(row['Local_PDF_URI']).strip()
        for _, row in df.iterrows()
        if str(row.get('Local_PDF_URI', '')).strip().startswith('file:///')
    }
    return local_links


local_pdf_links = _build_local_pdf_links_auto()
print(f"✔ Local downloaded PDF links loaded: {len(local_pdf_links)} valid file links found.")

if 'PDF_Status' in df.columns:
    print("\nPDF Status Summary:")
    print(df['PDF_Status'].value_counts().to_string())

if 'PDF_Match_Status' in df.columns:
    print("\nPDF Match Status Summary:")
    print(df['PDF_Match_Status'].value_counts().to_string())
    missing_preview = df[df['PDF_Status'] == 'Missing']
    if len(missing_preview):
        print("\nMissing / ambiguous local PDFs:")
        display_cols = ['Record Number', 'PDF_Match_Status']
        if 'Title' in df.columns:
            display_cols.append('Title')
        display(missing_preview[display_cols].head(30))

PATSEER_BASE = "https://app.patseer.com/#/search/result?q="

# ═══════════════════════════════════════════════════════════════
# COMMON PDF FOLDER LINK FOR SECTION 5
# ═══════════════════════════════════════════════════════════════
# All Record No. hyperlinks in Section 5 will point to this same folder.
#
# Option A — before uploading to Teams, keep the local relative folder:
#     PDF_FOLDER_LINK = "Downloaded_Patent_PDFs"
#
# Option B — after uploading the folder to Teams/SharePoint, paste the
# copied folder link here:
#     PDF_FOLDER_LINK = "https://yourcompany.sharepoint.com/sites/..."
#
# This avoids broken individual PDF links and lets users search the patent
# number manually inside the downloaded PDF folder.
# NOTE: the "?csf=1&web=1&e=..." share token is generated per-folder by
# SharePoint and can't be derived programmatically — verify this link still
# resolves after SharePoint upload each month, even though the month segment
# below now tracks MONTH_FOLDER automatically instead of being hand-typed.
# SharePoint folder naming uses Month_Year (e.g. "May_2026"), unlike the
# local-disk MONTH_FOLDER convention ("May2026") — derive it separately.
SHAREPOINT_MONTH_FOLDER = TARGET_MONTH.strftime("%B_%Y")
PDF_FOLDER_LINK = (
    "https://apollotyres.sharepoint.com/:f:/r/sites/GlobalPredevelopmentTeam/"
    "Shared%20Documents/Public/Monthly%20Patent%20Report%20-%20Pre%20Development/"
    f"{SHAREPOINT_MONTH_FOLDER}/Downloaded_Patent_PDFs?csf=1&web=1&e=gaHBZ6"
)

patseer_col  = ('Problem Being Solved (AI Sum.)' 
                if 'Problem Being Solved (AI Sum.)' in df.columns
                else 'Abstract')

print("✔ All dependencies present. Building report...")


# ════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════════

def set_cell_bg(cell, hex_color):
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd  = OxmlElement('w:shd')
    shd.set(qn('w:val'),   'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'),  hex_color)
    tcPr.append(shd)

def add_heading(doc, text, level=1, color='1F3864'):
    p = doc.add_heading(text, level=level)
    p.runs[0].font.color.rgb = RGBColor.from_string(color)
    return p

def add_caption(doc, text):
    p   = doc.add_paragraph(text)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.runs[0]
    run.font.size      = Pt(9)
    run.font.italic    = True
    run.font.color.rgb = RGBColor(100, 100, 100)

def set_table_full_width(table):
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit   = False
    tbl_pr = table._tbl.tblPr
    if tbl_pr is None:
        tbl_pr = OxmlElement('w:tblPr')
        table._tbl.insert(0, tbl_pr)
    tbl_w = tbl_pr.find(qn('w:tblW'))
    if tbl_w is None:
        tbl_w = OxmlElement('w:tblW')
        tbl_pr.append(tbl_w)
    tbl_w.set(qn('w:type'), 'pct')
    tbl_w.set(qn('w:w'),    '5000')
    tbl_layout = tbl_pr.find(qn('w:tblLayout'))
    if tbl_layout is None:
        tbl_layout = OxmlElement('w:tblLayout')
        tbl_pr.append(tbl_layout)
    tbl_layout.set(qn('w:type'), 'fixed')

def set_col_widths(table, widths_inches):
    set_table_full_width(table)
    available_inches = 6.5
    try:
        available_inches = float(
            (section.page_width - section.left_margin - section.right_margin) / 914400
        )
    except Exception:
        pass
    widths     = [float(w) for w in widths_inches]
    total_width = sum(widths)
    if total_width <= 0:
        return
    scaled = [w * available_inches / total_width for w in widths]
    for row in table.rows:
        for i, cell in enumerate(row.cells):
            if i < len(scaled):
                cell.width = Inches(scaled[i])

def get_patent_url(record_no):
    """
    Returns the common downloaded-PDF folder link for every patent card.
    Users should open the folder and search the patent number manually.

    If PDF_FOLDER_LINK is blank, falls back to the original PatSeer/search logic.
    """
    record_no = str(record_no).strip()

    folder_link = str(globals().get("PDF_FOLDER_LINK", "")).strip()
    if folder_link:
        return folder_link

    # Safety fallback only if common folder link is not configured
    if record_no in pdf_links:
        return pdf_links[record_no]
    return PATSEER_BASE + record_no

def add_hyperlink_to_cell(cell, display_text, url):
    paragraph = cell.paragraphs[0]
    paragraph.clear()
    part      = paragraph.part
    r_id      = part.relate_to(
        url,
        'http://schemas.openxmlformats.org/officeDocument/2006/relationships/hyperlink',
        is_external=True
    )
    hyperlink  = OxmlElement('w:hyperlink')
    hyperlink.set(qn('r:id'), r_id)
    run_elem   = OxmlElement('w:r')
    rPr        = OxmlElement('w:rPr')
    color_elem = OxmlElement('w:color')
    color_elem.set(qn('w:val'), '0563C1')
    rPr.append(color_elem)
    u_elem = OxmlElement('w:u')
    u_elem.set(qn('w:val'), 'single')
    rPr.append(u_elem)
    sz_elem = OxmlElement('w:sz')
    sz_elem.set(qn('w:val'), '16')
    rPr.append(sz_elem)
    run_elem.append(rPr)
    t_elem      = OxmlElement('w:t')
    t_elem.text = display_text
    run_elem.append(t_elem)
    hyperlink.append(run_elem)
    paragraph._p.append(hyperlink)

def add_label_value(cell, label, value, label_size=7, value_size=8):
    para = cell.paragraphs[0]
    para.clear()
    label_run = para.add_run(f"{label}\n")
    label_run.font.size      = Pt(label_size)
    label_run.font.bold      = True
    label_run.font.color.rgb = RGBColor(80, 80, 80)
    value_run = para.add_run(str(value) if str(value) != 'nan' else '—')
    value_run.font.size      = Pt(value_size)
    value_run.font.color.rgb = RGBColor(0, 0, 0)

def add_full_width_row(table, label, value,
                       label_hex='D6E4F0', value_hex='FFFFFF',
                       label_size=7, value_size=8):
    row   = table.add_row()
    cells = row.cells
    label_cell = cells[0]
    set_cell_bg(label_cell, label_hex)
    label_cell.paragraphs[0].clear()
    lr = label_cell.paragraphs[0].add_run(label)
    lr.font.size      = Pt(label_size)
    lr.font.bold      = True
    lr.font.color.rgb = RGBColor(50, 50, 50)
    value_cell = cells[1]
    for i in range(2, len(cells)):
        value_cell = value_cell.merge(cells[i])
    set_cell_bg(value_cell, value_hex)
    value_cell.paragraphs[0].clear()
    vr = value_cell.paragraphs[0].add_run(
        str(value) if str(value) not in ('nan', 'None', '') else '—'
    )
    vr.font.size      = Pt(value_size)
    vr.font.color.rgb = RGBColor(0, 0, 0)

def add_styled_table(doc, dataframe, header_hex='1F3864',
                     header_text_color='FFFFFF', font_size=8,
                     internal_link=False):
    HYPERLINK_COLS = ('Record Number', 'Record No.')
    table    = doc.add_table(rows=1 + len(dataframe), cols=len(dataframe.columns))
    table.style = 'Table Grid'
    set_table_full_width(table)
    col_names = list(dataframe.columns)

    for i, col_name in enumerate(col_names):
        cell = table.rows[0].cells[i]
        cell.text = str(col_name)
        set_cell_bg(cell, header_hex)
        run = cell.paragraphs[0].runs[0]
        run.font.bold      = True
        run.font.size      = Pt(font_size)
        run.font.color.rgb = RGBColor.from_string(header_text_color)
        cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

    for row_idx, (_, row_data) in enumerate(dataframe.iterrows()):
        row_color = 'FFFFFF' if row_idx % 2 == 0 else 'F2F2F2'
        for col_idx, value in enumerate(row_data):
            cell      = table.rows[row_idx + 1].cells[col_idx]
            value_str = '' if str(value) == 'nan' else str(value)
            set_cell_bg(cell, row_color)
            if col_names[col_idx] in HYPERLINK_COLS and value_str:
                if internal_link:
                    bookmark = 'patent_' + re.sub(r'[^a-zA-Z0-9]', '_', value_str.strip())
                    add_internal_hyperlink(cell, value_str, bookmark)
                else:
                    url = get_patent_url(value_str)
                    add_hyperlink_to_cell(cell, value_str, url)
            else:
                cell.text = value_str
                run = (cell.paragraphs[0].runs[0]
                       if cell.paragraphs[0].runs
                       else cell.paragraphs[0].add_run(value_str))
                run.font.size      = Pt(font_size)
                run.font.color.rgb = RGBColor(0, 0, 0)
    return table

# ── Summarisation ─────────────────────────────────────────────
print("✔ Using local summarisation (no API required)")


def clean_text_for_summary(text):
    """
    Cleans text for report summaries.
    """
    if not isinstance(text, str):
        return ""

    text = text.replace("\\n", " ").replace("\n", " ").strip()
    text = " ".join(text.split())
    return text


def trim_to_complete_sentence(text, max_chars=650):
    """
    Trims text without cutting abruptly in the middle of a sentence.
    Preferably ends at a complete sentence. If not possible, ends at a full word.
    """
    text = clean_text_for_summary(text)

    if not text:
        return "No summary available."

    if len(text) <= max_chars:
        return text

    trimmed = text[:max_chars]

    # Try to end at the last complete sentence punctuation
    last_sentence_end = max(
        trimmed.rfind("."),
        trimmed.rfind("?"),
        trimmed.rfind("!"),
        trimmed.rfind(";")
    )

    if last_sentence_end > 120:
        return trimmed[:last_sentence_end + 1].strip()

    # If no sentence end is found, end at the last complete word
    last_space = trimmed.rfind(" ")

    if last_space > 120:
        return trimmed[:last_space].rstrip() + "..."

    return trimmed.rstrip() + "..."


def local_extractive_summary(text, client=None):
    """
    Local fallback summary.
    Uses the first 3 sentences and trims without abrupt cutting.
    """
    text = clean_text_for_summary(text)

    if len(text) < 30:
        return "No summary available."

    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    if not sentences:
        return trim_to_complete_sentence(text, max_chars=650)

    summary = " ".join(sentences[:3])

    return trim_to_complete_sentence(summary, max_chars=650)


def get_notable_summary(row, client=None):
    """
    Priority:
    1) AI_Summary
    2) Problem Being Solved / selected PatSeer AI field
    3) Abstract
    4) Local fallback summary
    """
    ai_sum = clean_text_for_summary(str(row.get("AI_Summary", "")))

    if ai_sum and ai_sum not in ("nan", "None", "", "No summary available."):
        return trim_to_complete_sentence(ai_sum, max_chars=650)

    source_text = clean_text_for_summary(str(row.get(patseer_col, "")))

    if not source_text or source_text in ("nan", "None", ""):
        source_text = clean_text_for_summary(str(row.get("Abstract", "")))

    return local_extractive_summary(source_text)


def build_notable_df(subset_df, client=None):
    records = []

    for _, row in subset_df.iterrows():

        # Priority: AI_Summary > Condensed_Summary > PatSeer/Abstract fallback
        summary = clean_text_for_summary(str(row.get("AI_Summary", "")))

        if not summary or summary in ("nan", "None", "", "No summary available."):
            summary = clean_text_for_summary(str(row.get("Condensed_Summary", "")))

        # Safety for older typo/version if it exists
        if not summary or summary in ("nan", "None", "", "No summary available."):
            summary = clean_text_for_summary(str(row.get("Condensed_summary", "")))

        if not summary or summary in ("nan", "None", "", "No summary available."):
            summary = get_notable_summary(row)

        summary = trim_to_complete_sentence(summary, max_chars=650)

        novelty_val = row.get("Novelty_Index", np.nan)
        interest_val = row.get("Patent_Interest_Score", np.nan)

        records.append({
            "Record No.": str(row.get("Record Number", "")),
            "Assignee": str(row.get("Assignee_Normalized", "—")),
            "Novelty": "" if pd.isna(novelty_val) else f"{float(novelty_val):.1f}",
            "Interest Score": "" if pd.isna(interest_val) else f"{float(interest_val):.1f}",
            "Tire Region": str(row.get("Tire_Region", "—")),
            "Condensed Summary": summary,
        })

    return pd.DataFrame(records)

# ════════════════════════════════════════════════════════════════
# PDF DRAWING PAGE EXTRACTION — STABLE VERSION
# ════════════════════════════════════════════════════════════════
# Strategy:
#   1. Do NOT crop embedded images.
#   2. Do NOT extract individual raster objects.
#   3. Detect likely patent drawing pages.
#   4. Insert full-page snapshots only.
#
# This avoids:
#   - half-cut figures
#   - random image pieces
#   - cropped symbols/logos
#   - text blocks being treated as figures
# ════════════════════════════════════════════════════════════════

try:
    import fitz  # PyMuPDF
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyMuPDF'], check=True)
    import fitz

try:
    from PIL import Image
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'Pillow'], check=True)
    from PIL import Image

import os
import re
import glob
import hashlib
from urllib.parse import urlparse, unquote
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

# ── User settings ─────────────────────────────────────────────
INCLUDE_PDF_FIGURE_SNAPSHOTS = True

# Keep this small for report readability
MAX_FIGURE_SNAPSHOTS_PER_PATENT = 4

# 2.0 is good balance between clarity and file size
FIGURE_SNAPSHOT_DPI_ZOOM = 2.0

# Drawing page must pass this score
MIN_DRAWING_PAGE_SCORE = 4

FIGURE_SNAPSHOT_FOLDER = os.path.join(
    FILE_DIR if 'FILE_DIR' in globals() else os.getcwd(),
    "Extracted_PDF_Figures"
)

os.makedirs(FIGURE_SNAPSHOT_FOLDER, exist_ok=True)

PDF_FIGURE_CACHE = {}


def _file_uri_to_path(uri):
    """Convert file:/// URI to local Windows/Linux path."""
    if not isinstance(uri, str) or not uri.startswith("file:///"):
        return ""

    parsed = urlparse(uri)
    path = unquote(parsed.path)

    if os.name == "nt" and re.match(r"^/[A-Za-z]:/", path):
        path = path[1:]

    return path


def get_local_pdf_path(record_no):
    """Return local PDF path for a Record Number."""
    record_no = str(record_no).strip()

    if not record_no:
        return ""

    # Preferred: Local_PDF_Path column
    if "Local_PDF_Path" in df.columns and "Record Number" in df.columns:
        match = df[df["Record Number"].astype(str).str.strip() == record_no]
        if len(match):
            path = str(match.iloc[0].get("Local_PDF_Path", "")).strip()
            if path and path.lower() not in ("nan", "none") and os.path.exists(path):
                return path

    # Fallback: local_pdf_links if available
    if "local_pdf_links" in globals() and record_no in local_pdf_links:
        path = _file_uri_to_path(local_pdf_links[record_no])
        if path and os.path.exists(path):
            return path

    return ""


def _safe_record_folder_name(record_no):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(record_no).strip())[:80] or "unknown"


def _page_word_count(page):
    try:
        return len(page.get_text("words"))
    except Exception:
        return 9999


def _page_text_upper(page):
    try:
        return (page.get_text("text") or "").upper()
    except Exception:
        return ""


def _page_drawing_count(page):
    try:
        return len(page.get_drawings())
    except Exception:
        return 0


def _page_image_count(page):
    try:
        return len(page.get_images(full=True))
    except Exception:
        return 0


def drawing_page_score(page):
    """
    Score whether a page is likely a patent drawing page.

    We deliberately avoid cropping.
    We only decide whether the full page should be included.
    """

    score = 0

    word_count = _page_word_count(page)
    text_upper = _page_text_upper(page)
    drawing_count = _page_drawing_count(page)
    image_count = _page_image_count(page)

    # Reject clear text-heavy pages: description / claims / abstract
    if word_count > 180:
        return -10

    # Low-text pages are more likely to be drawings
    if word_count <= 30:
        score += 3
    elif word_count <= 80:
        score += 2
    elif word_count <= 130:
        score += 1

    # Patent figure labels
    if re.search(r"\bFIG\.?\s*\d+", text_upper):
        score += 4

    if re.search(r"\bFIGURE\s*\d+", text_upper):
        score += 3

    # Vector line drawings
    if drawing_count >= 30:
        score += 4
    elif drawing_count >= 10:
        score += 2
    elif drawing_count >= 3:
        score += 1

    # Raster drawing sheets
    if image_count >= 1:
        score += 1

    # Avoid cover/search report pages
    bad_text_markers = [
        "ABSTRACT",
        "CLAIMS",
        "DESCRIPTION",
        "INTERNATIONAL SEARCH REPORT",
        "WRITTEN OPINION",
        "REFERENCES CITED",
        "FIELD OF THE INVENTION",
        "BACKGROUND",
    ]

    # If page has these and no strong drawing signal, reject
    if any(marker in text_upper for marker in bad_text_markers):
        if drawing_count < 10 and not re.search(r"\bFIG\.?\s*\d+", text_upper):
            score -= 5

    return score


def _is_blank_or_nearly_blank_image(image_path):
    """
    Reject blank / near-white pages.
    """
    try:
        import numpy as np

        with Image.open(image_path) as im:
            gray = im.convert("L")
            arr = np.array(gray)

        # Very high mean = almost blank white page
        if arr.mean() > 252:
            return True

        # Very low variation also indicates blank/near blank
        if arr.std() < 3:
            return True

        return False

    except Exception:
        return False


def _save_full_page_snapshot(page, out_path):
    """
    Save full PDF page as PNG.
    No cropping.
    """
    matrix = fitz.Matrix(FIGURE_SNAPSHOT_DPI_ZOOM, FIGURE_SNAPSHOT_DPI_ZOOM)

    pix = page.get_pixmap(matrix=matrix, alpha=False)
    pix.save(out_path)
    pix = None

    try:
        with Image.open(out_path) as im:
            w, h = im.size

        if w < 500 or h < 500:
            os.remove(out_path)
            return False

        if _is_blank_or_nearly_blank_image(out_path):
            os.remove(out_path)
            return False

        return True

    except Exception:
        return False


def extract_pdf_figure_snapshots(record_no, pdf_path):
    """
    Extract full drawing-page snapshots from a local PDF.

    Output:
    - Full-page PNGs only
    - No cropped embedded images
    """
    record_no = str(record_no).strip()
    pdf_path = str(pdf_path).strip()

    if not pdf_path or not os.path.exists(pdf_path):
        return []

    cache_key = (
        record_no,
        os.path.abspath(pdf_path),
        MAX_FIGURE_SNAPSHOTS_PER_PATENT,
        FIGURE_SNAPSHOT_DPI_ZOOM,
        "full_page_only_v3"
    )

    if cache_key in PDF_FIGURE_CACHE:
        cached = PDF_FIGURE_CACHE[cache_key]
        if all(os.path.exists(p) for p in cached):
            return cached

    rec_folder = os.path.join(
        FIGURE_SNAPSHOT_FOLDER,
        _safe_record_folder_name(record_no)
    )
    os.makedirs(rec_folder, exist_ok=True)

    # Clear old outputs for this patent
    for old_png in glob.glob(os.path.join(rec_folder, "*.png")):
        try:
            os.remove(old_png)
        except Exception:
            pass

    try:
        pdf = fitz.open(pdf_path)
    except Exception as e:
        print(f"⚠ Cannot open PDF for {record_no}: {e}")
        PDF_FIGURE_CACHE[cache_key] = []
        return []

    selected_pages = []

    try:
        for page_idx in range(len(pdf)):
            page = pdf[page_idx]
            score = drawing_page_score(page)

            if score >= MIN_DRAWING_PAGE_SCORE:
                selected_pages.append((page_idx, score))

        # Preserve document order
        selected_pages = sorted(selected_pages, key=lambda x: x[0])

        # Limit number of drawing pages
        if MAX_FIGURE_SNAPSHOTS_PER_PATENT is not None:
            selected_pages = selected_pages[:MAX_FIGURE_SNAPSHOTS_PER_PATENT]

        out_files = []
        seen_hashes = set()

        for page_idx, score in selected_pages:
            page = pdf[page_idx]

            out_path = os.path.join(
                rec_folder,
                f"drawing_page_{page_idx + 1:03d}_score_{score}.png"
            )

            ok = _save_full_page_snapshot(page, out_path)

            if not ok:
                continue

            # Deduplicate identical snapshots
            try:
                with open(out_path, "rb") as f:
                    digest = hashlib.md5(f.read()).hexdigest()

                if digest in seen_hashes:
                    os.remove(out_path)
                    continue

                seen_hashes.add(digest)

            except Exception:
                pass

            out_files.append(out_path)

    except Exception as e:
        print(f"⚠ Drawing extraction warning for {record_no}: {e}")
        out_files = []

    finally:
        try:
            pdf.close()
        except Exception:
            pass

    PDF_FIGURE_CACHE[cache_key] = out_files
    return out_files


def add_pdf_figure_snapshots_to_doc(doc, record_no):
    """
    Insert full drawing-page snapshots below the respective patent card.
    Scoped to NOTABLE_RECORD_NUMBERS (see Cell 6F) — at full monthly
    volume, bulk-downloading and processing a PDF for every single patent
    isn't practical, so only the patents already surfaced in Section 4
    (Top Interest Score, Top Novelty, Cross-domain, AI/ML, Sustainability,
    Smart/Connected) get drawing pages. Every patent still gets a full
    text card regardless.
    """
    if not INCLUDE_PDF_FIGURE_SNAPSHOTS:
        return

    if record_no not in NOTABLE_RECORD_NUMBERS:
        p = doc.add_paragraph(
            "PDF drawings: not a notable patent this month (see Section 4) — "
            "drawings omitted to keep monthly PDF handling manageable at current volume."
        )
        run = p.runs[0]
        run.font.size = Pt(8)
        run.font.italic = True
        return

    pdf_path = get_local_pdf_path(record_no)

    if not pdf_path:
        p = doc.add_paragraph("PDF drawings: local file not found — drawings not inserted.")
        run = p.runs[0]
        run.font.size = Pt(8)
        run.font.italic = True
        return

    figure_paths = extract_pdf_figure_snapshots(record_no, pdf_path)

    if not figure_paths:
        p = doc.add_paragraph("PDF drawings: no clean drawing pages detected.")
        run = p.runs[0]
        run.font.size = Pt(8)
        run.font.italic = True
        return

    p = doc.add_paragraph()
    run = p.add_run(f"PDF drawing pages ({len(figure_paths)} shown)")
    run.bold = True
    run.font.size = Pt(9)
    run.font.color.rgb = RGBColor.from_string("1F3864")

    # One full page per row — avoids tiny unreadable grids
    fig_table = doc.add_table(rows=len(figure_paths), cols=1)
    fig_table.style = "Table Grid"
    set_table_full_width(fig_table)

    for i, img_path in enumerate(figure_paths):
        cell = fig_table.rows[i].cells[0]
        cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

        try:
            cell.paragraphs[0].add_run().add_picture(img_path, width=Inches(5.8))
        except Exception as e:
            cell.text = f"Could not insert drawing page: {os.path.basename(img_path)} ({e})"

    doc.add_paragraph()

# ════════════════════════════════════════════════════════════════
# BUILD DOCUMENT
# ════════════════════════════════════════════════════════════════
doc     = Document()
section = doc.sections[0]
for margin in ('top_margin', 'bottom_margin', 'left_margin', 'right_margin'):
    setattr(section, margin, Inches(1))

# ── Page Numbers in Footer ────────────────────────────────────
def add_page_numbers(doc):
    section = doc.sections[0]
    footer  = section.footer
    footer.is_linked_to_previous = False
    para = footer.paragraphs[0] if footer.paragraphs else footer.add_paragraph()
    para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    para.clear()

    run = para.add_run("Page ")
    run.font.size      = Pt(9)
    run.font.color.rgb = RGBColor(100, 100, 100)

    fld_char1 = OxmlElement('w:fldChar')
    fld_char1.set(qn('w:fldCharType'), 'begin')
    instr = OxmlElement('w:instrText')
    instr.set(qn('xml:space'), 'preserve')
    instr.text = 'PAGE'
    fld_char2 = OxmlElement('w:fldChar')
    fld_char2.set(qn('w:fldCharType'), 'end')
    run_pg = OxmlElement('w:r')
    run_pg.append(fld_char1)
    run_pg.append(instr)
    run_pg.append(fld_char2)
    para._p.append(run_pg)

    run2 = para.add_run(" of ")
    run2.font.size      = Pt(9)
    run2.font.color.rgb = RGBColor(100, 100, 100)

    fld_char3 = OxmlElement('w:fldChar')
    fld_char3.set(qn('w:fldCharType'), 'begin')
    instr2 = OxmlElement('w:instrText')
    instr2.set(qn('xml:space'), 'preserve')
    instr2.text = 'NUMPAGES'
    fld_char4 = OxmlElement('w:fldChar')
    fld_char4.set(qn('w:fldCharType'), 'end')
    run_np = OxmlElement('w:r')
    run_np.append(fld_char3)
    run_np.append(instr2)
    run_np.append(fld_char4)
    para._p.append(run_np)

    left_run = para.add_run(f"  |  {REPORT_TITLE}")
    left_run.font.size      = Pt(8)
    left_run.font.italic    = True
    left_run.font.color.rgb = RGBColor(150, 150, 150)

add_page_numbers(doc)

# ── Title Page ────────────────────────────────────────────────
doc.add_paragraph()
doc.add_paragraph()

title_para = doc.add_paragraph()
title_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
tr = title_para.add_run(REPORT_TITLE)
tr.font.size      = Pt(24)
tr.font.bold      = True
tr.font.color.rgb = RGBColor.from_string('1F3864')

doc.add_paragraph()
week_para = doc.add_paragraph()
week_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
wr = week_para.add_run(
    f"{MONTH_NAME}  |  "
    f"{df['Publication/Issue Date'].min().strftime('%d %b %Y')} – "
    f"{df['Publication/Issue Date'].max().strftime('%d %b %Y')}"
)
wr.font.size      = Pt(14)
wr.font.color.rgb = RGBColor(80, 80, 80)

doc.add_paragraph()
auth_para = doc.add_paragraph()
auth_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
ar = auth_para.add_run(f"Generated by: {AUTHOR}")
ar.font.size    = Pt(12)
ar.font.italic  = True

date_para = doc.add_paragraph()
date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
dr = date_para.add_run(f"Report Date: {datetime.now().strftime('%d %B %Y')}")
dr.font.size      = Pt(11)
dr.font.color.rgb = RGBColor(100, 100, 100)

# ── Company Information Block ─────────────────────────────────
COMPANY_NAME    = "Apollo Global R&D."
COMPANY_DEPT    = "Technology Pre-Development"
COMPANY_ADDRESS = "Chennai, Tamil Nadu, India"

doc.add_paragraph()
doc.add_paragraph()

divider = doc.add_paragraph('─' * 60)
divider.alignment = WD_ALIGN_PARAGRAPH.CENTER
divider.runs[0].font.color.rgb = RGBColor(180, 180, 180)

doc.add_paragraph()

co_para = doc.add_paragraph()
co_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
co_run = co_para.add_run(COMPANY_NAME)
co_run.font.size      = Pt(13)
co_run.font.bold      = True
co_run.font.color.rgb = RGBColor.from_string('1F3864')

dept_para = doc.add_paragraph()
dept_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
dept_run = dept_para.add_run(COMPANY_DEPT)
dept_run.font.size      = Pt(11)
dept_run.font.color.rgb = RGBColor(80, 80, 80)

addr_para = doc.add_paragraph()
addr_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
addr_run = addr_para.add_run(COMPANY_ADDRESS)
addr_run.font.size      = Pt(10)
addr_run.font.color.rgb = RGBColor(120, 120, 120)

doc.add_page_break()

# ── Table of Contents ─────────────────────────────────────────
toc_para = doc.add_paragraph('Table of Contents')
toc_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
toc_run = toc_para.runs[0]
toc_run.font.size      = Pt(13)
toc_run.font.bold      = True
toc_run.font.color.rgb = RGBColor.from_string('1F3864')
doc.add_paragraph()

toc_entries = [
    '1.  Executive Summary',
    '2.  Summary Statistics',
    '    2.1  Technology Levers Distribution',
    '    2.2  Performance Targets Distribution',
    '    2.3  Tyre Segment Distribution',
    '    2.4  Top Assignees',
    '3.  Visual Analytics',
    '    Figure 1 – Technology Levers vs Performance Targets Heatmap',
    '4.  Notable Patents',
    '    4.0  Notable Patents Shortlist (PDF-detailed)',
    '    4.1  Patent Distribution by Tire Region',
    '    4.2  Top 5 Interesting Patents of the Month',
    '    4.3  Highest Novelty Patents',
    '    4.4  Cross-Domain Patents (≥3 Performance Targets)',
    '    4.5  AI / ML Patents',
    '    4.6  Sustainability Patents',
    '    4.7  Smart / Connected Tire Patents',
    '5.  Full Patent List',
]
toc_table = doc.add_table(rows=len(toc_entries), cols=1)
toc_table.style = 'Table Grid'
toc_table.alignment = WD_TABLE_ALIGNMENT.CENTER
for i, entry in enumerate(toc_entries):
    cell = toc_table.rows[i].cells[0]
    cell.text = entry
    set_cell_bg(cell, 'EAF0FB' if i % 2 == 0 else 'FFFFFF')
    run      = cell.paragraphs[0].runs[0]
    is_main  = entry.startswith(('1.', '2.', '3.', '4.', '5.'))
    run.font.size      = Pt(11 if is_main else 10)
    run.font.bold      = is_main
    run.font.color.rgb = (RGBColor.from_string('1F3864')
                          if is_main else RGBColor(80, 80, 80))
doc.add_page_break()


# ════════════════════════════════════════════════════════════════
# SECTION 1: EXECUTIVE SUMMARY
# ════════════════════════════════════════════════════════════════
add_heading(doc, '1. Executive Summary')

top3_assignees = (df[~df['Assignee_Normalized'].isin(['Others', 'Other Tire/Rubber Manufacturer'])]
                  ['Assignee_Normalized'].value_counts().head(3))
top3_countries = df['Publication Country'].value_counts().head(3)
top1_tl        = count_labels(df['Technology_Levers']).iloc[0]
top1_pt        = count_labels(df['Performance_Targets']).iloc[0]

for text in [
    (f"This report covers the monthly patent analysis for {MONTH_NAME}. "
     f"{TOTAL_PUBLICATIONS} patent publications were retrieved, representing "
     f"{len(df)} unique inventions (patent families) after deduplicating "
     f"publications of the same invention across jurisdictions "
     f"({TOTAL_PUBLICATIONS - len(df)} duplicate publication(s) collapsed). "
     f"Publications were dated between "
     f"{all_publications_df['Publication/Issue Date'].min().strftime('%d %b %Y')} and "
     f"{all_publications_df['Publication/Issue Date'].max().strftime('%d %b %Y')}."),
    (f"The leading assignees were "
     f"{', '.join([f'{k} ({v} patents)' for k, v in top3_assignees.items()])}. "
     f"The most active publication countries were "
     f"{', '.join([f'{k} ({v})' for k, v in top3_countries.items()])}."),
    (f"The dominant technology lever was {top1_tl['Label']} "
     f"({top1_tl['Count']} patents), and the most targeted performance "
     f"attribute was {top1_pt['Label']} ({top1_pt['Count']} patents).")
]:
    p = doc.add_paragraph(text)
    p.runs[0].font.size = Pt(10)

if str(globals().get('SCOPE_CHANGE_NOTE', '')).strip():
    _note_p = doc.add_paragraph()
    _note_run = _note_p.add_run(SCOPE_CHANGE_NOTE)
    _note_run.font.size   = Pt(9)
    _note_run.font.italic  = True
    _note_run.font.color.rgb = RGBColor(120, 120, 120)

doc.add_page_break()


# ════════════════════════════════════════════════════════════════
# SECTION 2: SUMMARY STATISTICS
# ════════════════════════════════════════════════════════════════
add_heading(doc, '2. Summary Statistics')

add_heading(doc, '2.1 Technology Levers Distribution', level=2)
tl_tbl = count_labels(df['Technology_Levers'])
tl_tbl.columns = ['Technology Lever', 'Patent Count']
tbl = add_styled_table(doc, tl_tbl, font_size=10)
set_col_widths(tbl, [3.5, 2.0])
doc.add_paragraph()

add_heading(doc, '2.2 Performance Targets Distribution', level=2)
pt_tbl = count_labels(df['Performance_Targets'])
pt_tbl.columns = ['Performance Target', 'Patent Count']
tbl = add_styled_table(doc, pt_tbl, font_size=10)
set_col_widths(tbl, [3.5, 2.0])
doc.add_paragraph()

add_heading(doc, '2.3 Tyre Segment Distribution', level=2)
p = doc.add_paragraph(
    "Classification of patents by tyre segment based on keyword matching. "
    "'Unspecified' indicates patents with no segment-specific language detected."
)
p.runs[0].font.size = Pt(9)

SEGMENT_ORDER = ['PCR', 'TBR', 'Off_Highway', 'Two_Wheeler', 'Specialty', 'Unspecified']
seg_counts    = {seg: 0 for seg in SEGMENT_ORDER}
for seg_val in df['Segment'].fillna('Unspecified').astype(str):
    parts   = [s.strip() for s in seg_val.split('|')]
    matched = False
    for part in parts:
        if part in seg_counts:
            seg_counts[part] += 1
            matched = True
    if not matched:
        seg_counts['Unspecified'] += 1

seg_tbl = pd.DataFrame([
    {'Tyre Segment': seg, 'Patent Count': seg_counts[seg]}
    for seg in SEGMENT_ORDER
])
tbl = add_styled_table(doc, seg_tbl, font_size=10)
set_col_widths(tbl, [3.5, 2.0])
doc.add_paragraph()

add_heading(doc, '2.4 Top Assignees', level=2)
asgn_tbl = (df[~df['Assignee_Normalized'].isin(['Others', 'Other Tire/Rubber Manufacturer'])]
            ['Assignee_Normalized'].value_counts().reset_index())
asgn_tbl.columns = ['Assignee', 'Patent Count']
tbl = add_styled_table(doc, asgn_tbl, font_size=10)
set_col_widths(tbl, [3.5, 2.0])

doc.add_page_break()


# ════════════════════════════════════════════════════════════════
# SECTION 3: VISUAL ANALYTICS
# ════════════════════════════════════════════════════════════════
add_heading(doc, '3. Visual Analytics')
heatmap_path = os.path.join(FILE_DIR, 'chart_02_heatmap_pt_tl_viridis_readable.png')
if os.path.exists(heatmap_path):
    doc.add_picture(heatmap_path, width=Inches(6.0))
    add_caption(doc,
        'Figure 1: Technology Levers vs Performance Targets '
        'Heatmap (Patent Co-occurrence Count)')
else:
    doc.add_paragraph("⚠ Heatmap not found. Run Cell 10 first.")
doc.add_page_break()


# ════════════════════════════════════════════════════════════════
# SECTION 4: NOTABLE PATENTS
# ════════════════════════════════════════════════════════════════

def _cap_section4(subset_df, doc, n=SECTION4_MAX_ROWS_PER_TABLE):
    """Rank a thematic subset by Interest Score and keep top-n; if it was
    truncated, add an italic note pointing to the full list in the Excel."""
    total = len(subset_df)
    capped = (subset_df
              .sort_values(['Patent_Interest_Score', 'Novelty_Index'], ascending=False)
              .head(n))
    if total > n:
        p = doc.add_paragraph(
            f"Showing top {n} of {total} by Interest Score — full list in the classified Excel."
        )
        run = p.runs[0]
        run.font.size = Pt(8)
        run.font.italic = True
    return capped

# Tight Smart/Connected signal for Section 4.7 — the broad
# Technology_Levers 'Smart' / Performance_Targets 'Smart_Connected'
# buckets (generic 'sensor'/'wireless'/'transmission' + the B60C23 CPC
# merge) matched ~28% of all patents, which is too loose for a highlight
# table. This requires an explicit smart-tire term in title/abstract/claims.
SMART_CONNECTED_STRONG_TERMS = [
    'tpms', 'tire pressure monitoring', 'tyre pressure monitoring',
    'rfid', 'transponder',
    'smart tire', 'smart tyre', 'connected tire', 'connected tyre',
    'intelligent tire', 'intelligent tyre',
    'embedded sensor', 'tire sensor', 'tyre sensor', 'in-tire sensor', 'in tire sensor',
    'strain sensor', 'wireless sensor',
    'energy harvesting', 'piezoelectric', 'triboelectric', 'self-powered', 'self powered',
    'telematics',
    'tread wear monitoring', 'tire wear monitoring', 'tyre wear monitoring',
]

def _is_smart_connected(row):
    text = (str(row.get('Title_lower', '')) + ' ' +
            str(row.get('Abstract_lower', '')) + ' ' +
            str(row.get('Claims_lower', '')))
    return any(term in text for term in SMART_CONNECTED_STRONG_TERMS)

add_heading(doc, '4. Notable Patents')
doc.add_paragraph(
    'This section highlights the most notable patents for the month based on overall interest score, '
    'novelty, cross-domain coverage, AI/ML content, sustainability focus, and smart/connected technology. '
    'Click any Record No. to jump to the full patent card in Section 5. In Section 5, notable patents\' '
    'Record No. opens the shared downloaded-PDF folder (search the patent number inside it); other cards '
    'are unlinked since only notable-patent PDFs are downloaded at current volume.'
).runs[0].font.size = Pt(10)

# 4.0 Notable Patents Shortlist — the consolidated set that gets PDF
# drawings + folder links in Section 5. Listed here in one place so these
# can be identified without hunting through the full 564-card Section 5;
# each Record No. links straight to its Section 5 card.
add_heading(doc, 'Notable Patents Shortlist (PDF-detailed)', level=2)
doc.add_paragraph(
    f"The {len(NOTABLE_RECORD_NUMBERS)} patents below are this month's shortlist — the only ones "
    f"given PDF drawing pages and a downloaded-PDF-folder link in Section 5 (capped at "
    f"MAX_NOTABLE_PATENTS). 'Why Notable' shows which highlight criteria each one met. "
    f"Click any Record No. to jump to its full card in Section 5."
).runs[0].font.size = Pt(10)

def _why_notable(rec):
    rec = str(rec)
    tags = []
    if rec in _top5_interest_records: tags.append('Top Interest')
    if rec in _top5_novelty_records:  tags.append('Top Novelty')
    if rec in _cross_domain_records:  tags.append('Cross-domain')
    if rec in _aiml_records:          tags.append('AI/ML')
    return ', '.join(tags) if tags else 'High Interest Score'

_shortlist = (
    df[df['Is_Notable_Patent']]
    .sort_values(['Patent_Interest_Score', 'Novelty_Index'], ascending=False)
    .copy()
)
shortlist_tbl = pd.DataFrame({
    'Record No.':     _shortlist['Record Number'].astype(str),
    'Assignee':       _shortlist['Assignee_Normalized'].astype(str),
    'Segment':        _shortlist['Segment'].astype(str),
    'Interest Score': _shortlist['Patent_Interest_Score'].map(lambda v: f"{float(v):.1f}"),
    'Novelty':        _shortlist['Novelty_Index'].map(lambda v: f"{float(v):.1f}"),
    'Why Notable':    _shortlist['Record Number'].map(_why_notable),
})
tbl = add_styled_table(doc, shortlist_tbl, font_size=8, internal_link=True)
set_col_widths(tbl, [1.2, 1.4, 1.0, 0.9, 0.8, 1.5])
doc.add_paragraph()

# 4.1 Patent distribution by tire region
region_dist = count_labels(df['Tire_Region']).copy()
add_heading(doc, '4.1 Patent Distribution by Tire Region', level=2)
if len(region_dist):
    region_dist = region_dist.rename(columns={'Label': 'Tire Region', 'Count': 'Patent Count'})
    region_dist['Percentage (%)'] = ((region_dist['Patent Count'] / max(len(df), 1)) * 100).round(1)
    doc.add_paragraph(
        'Distribution of the month\'s patents across tire regions based on keyword classification. '
        'A patent may appear in more than one region when multiple structural areas are discussed.'
    ).runs[0].font.size = Pt(10)
    tbl = add_styled_table(doc, region_dist, font_size=9)
    set_col_widths(tbl, [2.5, 1.5, 1.5])
    doc.add_paragraph()
else:
    doc.add_paragraph('No tire-region labels were identified in the current dataset.').runs[0].font.italic = True
    doc.add_paragraph()

doc.add_paragraph(
    'Novelty is calculated from the rarity of Technology Lever × Performance Target combinations inside the '
    'current month dataset. For each patent, all TL–PT pairs are generated, each pair receives a rarity score '
    'of 1 / frequency, the patent receives the average of those rarity scores, and the result is min-max '
    'normalized to a 0–100 scale.'
).runs[0].font.size = Pt(9)

doc.add_paragraph(
    'Interest Score is a transparent weighted score on a 0–100 scale: 45% Novelty Index + 20% Technology Lever '
    'breadth + 20% Performance Target breadth + 10% Tire Region breadth + 5% cross-functional bonus. Higher '
    'scores indicate patents that are both technically uncommon and broad in potential relevance.'
).runs[0].font.size = Pt(9)

# 4.2 Top 5 Interesting Patents
add_heading(doc, '4.2 Top 5 Interesting Patents of the Month', level=2)
doc.add_paragraph(
    'Highest-ranked patents using the Interest Score. These patents combine uncommon TL–PT combinations with '
    'broader technology/performance coverage and clearer tire-region relevance.'
).runs[0].font.size = Pt(10)
top5_report_df = (
    top5_patents.loc[:, [
        'Record Number', 'Assignee_Normalized', 'Patent_Interest_Score',
        'Novelty_Index', 'Tire_Region', 'Interest_Reason'
    ]]
    .rename(columns={
        'Record Number':         'Record No.',
        'Assignee_Normalized':   'Assignee',
        'Patent_Interest_Score': 'Interest Score',
        'Novelty_Index':         'Novelty',
        'Tire_Region':           'Tire Region',
        'Interest_Reason':       'Why Interesting'
    })
)
tbl = add_styled_table(doc, top5_report_df, font_size=8, internal_link=True)
set_col_widths(tbl, [1.1, 1.3, 1.0, 0.9, 1.3, 3.0])
doc.add_paragraph()

# 4.3 Highest Novelty Patents
add_heading(doc, '4.3 Highest Novelty Patents', level=2)
doc.add_paragraph(
    'Patents with the rarest Technology Lever × Performance Target combinations in the current month dataset.'
).runs[0].font.size = Pt(10)
novelty_report_df = (
    df.sort_values(['Novelty_Index', 'Patent_Interest_Score'], ascending=[False, False])
      .loc[:, ['Record Number', 'Assignee_Normalized', 'Novelty_Index',
               'Patent_Interest_Score', 'Tire_Region', 'Technology_Levers']]
      .head(5)
      .rename(columns={
          'Record Number':         'Record No.',
          'Assignee_Normalized':   'Assignee',
          'Novelty_Index':         'Novelty',
          'Patent_Interest_Score': 'Interest Score',
          'Tire_Region':           'Tire Region',
          'Technology_Levers':     'Technology Levers'
      })
)
tbl = add_styled_table(doc, novelty_report_df, font_size=8, internal_link=True)
set_col_widths(tbl, [1.1, 1.3, 0.9, 1.0, 1.3, 3.0])
doc.add_paragraph()

# 4.4 Cross-Domain Patents
add_heading(doc, '4.4 Cross-Domain Patents (≥3 Performance Targets)', level=2)
doc.add_paragraph(
    'Patents addressing three or more performance targets simultaneously — indicative of broad or platform-level innovations.'
).runs[0].font.size = Pt(10)
cross_subset = df[df['PT_Count'] >= 3].copy()
if len(cross_subset):
    print(f"  Building table for {len(cross_subset)} cross-domain patents (capped at {SECTION4_MAX_ROWS_PER_TABLE})...")
    cross_rpt = build_notable_df(_cap_section4(cross_subset, doc))
    tbl = add_styled_table(doc, cross_rpt, font_size=8, internal_link=True)
    set_col_widths(tbl, [1.0, 1.2, 0.8, 1.0, 1.3, 3.2])
else:
    doc.add_paragraph('No cross-domain patents identified.').runs[0].font.italic = True
doc.add_paragraph()

# 4.5 AI/ML Patents
add_heading(doc, '4.5 AI/ML Patents', level=2)
doc.add_paragraph(
    'Patents containing AI, machine learning, neural networks, or data-driven techniques in tire technology.'
).runs[0].font.size = Pt(10)
aiml_subset = df[df['Contains_AI_ML']].copy()
if len(aiml_subset):
    print(f"  Building table for {len(aiml_subset)} AI/ML patents (capped at {SECTION4_MAX_ROWS_PER_TABLE})...")
    aiml_rpt = build_notable_df(_cap_section4(aiml_subset, doc))
    tbl = add_styled_table(doc, aiml_rpt, font_size=8, internal_link=True)
    set_col_widths(tbl, [1.0, 1.2, 0.8, 1.0, 1.3, 3.2])
else:
    doc.add_paragraph('No AI/ML patents identified.').runs[0].font.italic = True
doc.add_paragraph()

# 4.6 Sustainability Patents
add_heading(doc, '4.6 Sustainability Patents', level=2)
doc.add_paragraph(
    'Patents focused on sustainable materials, recycled content, bio-based compounds, or reduced environmental impact.'
).runs[0].font.size = Pt(10)
sustain_subset = df[df['Performance_Targets'].str.contains('Sustainability', na=False)].copy()
if len(sustain_subset):
    print(f"  Building table for {len(sustain_subset)} sustainability patents (capped at {SECTION4_MAX_ROWS_PER_TABLE})...")
    sustain_rpt = build_notable_df(_cap_section4(sustain_subset, doc))
    tbl = add_styled_table(doc, sustain_rpt, font_size=8, internal_link=True)
    set_col_widths(tbl, [1.0, 1.2, 0.8, 1.0, 1.3, 3.2])
else:
    doc.add_paragraph('No sustainability patents identified.').runs[0].font.italic = True
doc.add_paragraph()

# 4.7 Smart/Connected Tire Patents
add_heading(doc, '4.7 Smart/Connected Tire Patents', level=2)
doc.add_paragraph(
    'Patents with an explicit smart-tire signal — TPMS, RFID/transponder, embedded tire sensors, energy harvesting, or wear monitoring.'
).runs[0].font.size = Pt(10)
smart_subset = df[df.apply(_is_smart_connected, axis=1)].copy()
if len(smart_subset):
    print(f"  Building table for {len(smart_subset)} smart/connected patents (capped at {SECTION4_MAX_ROWS_PER_TABLE})...")
    smart_rpt = build_notable_df(_cap_section4(smart_subset, doc))
    tbl = add_styled_table(doc, smart_rpt, font_size=8, internal_link=True)
    set_col_widths(tbl, [1.0, 1.2, 0.8, 1.0, 1.3, 3.2])
else:
    doc.add_paragraph('No smart/connected patents identified.').runs[0].font.italic = True

doc.add_page_break()
print("✔ Section 4 complete.")


# ════════════════════════════════════════════════════════════════
# SECTION 5: FULL PATENT LIST — CARD LAYOUT
# ════════════════════════════════════════════════════════════════
HEADER_HEX    = '5B9BD5'
SUBHEADER_HEX = '2E5FA3'
LABEL_HEX     = 'D6E4F0'
WHITE         = 'FFFFFF'
LIGHT_GREY    = 'F5F5F5'

def _budget_card_text(text, max_chars):
    """Trim a free-text card field to a character budget at a sentence
    boundary (falling back to word boundary), so a card stays within one
    page. Empty/placeholder values pass through as an em dash. The full
    untrimmed text always remains in the classified Excel export."""
    # Collapse all runs of whitespace (including the hard line breaks that
    # claims text is full of) to single spaces, so the character budget
    # reliably predicts rendered height and a card can't blow past a page
    # just because its claims contain many newlines.
    t = " ".join(str(text).split())
    if not t or t.lower() in ('nan', 'none', ''):
        return '—'
    if len(t) <= max_chars:
        return t
    trimmed = trim_to_complete_sentence(t, max_chars=max_chars)
    return trimmed + ' […full text in classified Excel]'


def _prevent_card_row_splits(table):
    """Mark every row of a card table 'cannot split across pages' so a
    single row never straddles a page break."""
    for row in table.rows:
        trPr = row._tr.get_or_add_trPr()
        trPr.append(OxmlElement('w:cantSplit'))


def add_patent_card(doc, row_data, card_number):
    # Each patent card starts on a fresh page so no card is split across a
    # page boundary; combined with the text budget above, a card fits one
    # page. (Notable-patent drawing pages intentionally follow on the next
    # pages — the one-page rule is about the patent table itself.)
    if SECTION5_ONE_CARD_PER_PAGE:
        doc.add_page_break()

    record_no     = str(row_data.get('Record Number', '')).strip()
    title         = str(row_data.get('Title', '-'))[:150]
    bookmark_name = 'patent_' + re.sub(r'[^a-zA-Z0-9]', '_', record_no)
    # Only notable patents (Cell 6F) get the shared-PDF-folder hyperlink —
    # that folder only contains PDFs for the notable set, so linking every
    # card there would send readers to a folder that lacks their patent.
    is_notable    = record_no in NOTABLE_RECORD_NUMBERS
    pdf_url       = get_patent_url(record_no) if is_notable else ""

    table = doc.add_table(rows=0, cols=4)
    table.style = 'Table Grid'
    set_table_full_width(table)

    # Row 1: Card number | Record No. (common PDF folder link) | Title
    r0     = table.add_row()
    cells0 = r0.cells
    set_cell_bg(cells0[0], HEADER_HEX)
    cells0[0].paragraphs[0].clear()
    nr = cells0[0].paragraphs[0].add_run(f"#{card_number}")
    nr.font.size      = Pt(9)
    nr.font.bold      = True
    nr.font.color.rgb = RGBColor(255, 255, 255)
    cells0[0].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

    set_cell_bg(cells0[1], HEADER_HEX)
    if pdf_url:
        add_hyperlink_to_cell(cells0[1], record_no, pdf_url)
        # Change patent number hyperlink color in Section 5
        for h in cells0[1]._tc.xpath('.//w:hyperlink'):
            for color in h.xpath('.//w:color'):
                color.set(qn('w:val'), 'FFFFFF')   # white font
    else:
        cells0[1].paragraphs[0].clear()
        cells0[1].paragraphs[0].add_run(record_no)
    add_bookmark(cells0[1].paragraphs[0], bookmark_name)
    for para in cells0[1].paragraphs:
        for run in para.runs:
            run.font.size      = Pt(9)
            run.font.bold      = True
            run.font.color.rgb = RGBColor(255, 255, 255)

    title_cell = cells0[2].merge(cells0[3])
    set_cell_bg(title_cell, HEADER_HEX)
    title_cell.paragraphs[0].clear()
    tr = title_cell.paragraphs[0].add_run(title)
    tr.font.size      = Pt(9)
    tr.font.bold      = True
    tr.font.color.rgb = RGBColor(255, 255, 255)

    # Row 2: Assignee | Pub Date | Country
    r1     = table.add_row()
    cells1 = r1.cells
    set_cell_bg(cells1[0], LIGHT_GREY)
    add_label_value(cells1[0], 'Assignee', row_data.get('Assignee_Normalized', '—'))
    set_cell_bg(cells1[1], LIGHT_GREY)
    pub_date = row_data.get('Publication/Issue Date', '')
    if hasattr(pub_date, 'strftime'):
        pub_date = pub_date.strftime('%d %b %Y')
    add_label_value(cells1[1], 'Pub. Date', pub_date)
    set_cell_bg(cells1[2], LIGHT_GREY)
    add_label_value(cells1[2], 'Country', row_data.get('Publication Country', '—'))
    set_cell_bg(cells1[3], LIGHT_GREY)
    add_label_value(cells1[3], 'Segment', row_data.get('Segment', '—'))

    # Row 3: Performance Targets | Technology Levers
    r2     = table.add_row()
    cells2 = r2.cells
    set_cell_bg(cells2[0], LIGHT_GREY)
    add_label_value(cells2[0], 'Segment', row_data.get('Segment', '—'))
    set_cell_bg(cells2[1], LIGHT_GREY)
    add_label_value(cells2[1], 'Performance Targets', row_data.get('Performance_Targets', '—'))
    tl_cell = cells2[2].merge(cells2[3])
    set_cell_bg(tl_cell, LIGHT_GREY)
    add_label_value(tl_cell, 'Technology Levers', row_data.get('Technology_Levers', '—'))

    # Row 4: Tire Region | Breadth Band | Novelty | Interest Score
    r3     = table.add_row()
    cells3 = r3.cells
    set_cell_bg(cells3[0], LIGHT_GREY)
    add_label_value(cells3[0], 'Tire Region', row_data.get('Tire_Region', '—'))
    set_cell_bg(cells3[1], LIGHT_GREY)
    add_label_value(cells3[1], 'Breadth Band', row_data.get('Breadth_Band', '—'))
    set_cell_bg(cells3[2], LIGHT_GREY)
    novelty_val  = row_data.get('Novelty_Index', '—')
    novelty_text = f"{novelty_val}" if novelty_val == '—' else f"{float(novelty_val):.1f}"
    add_label_value(cells3[2], 'Novelty', novelty_text)
    set_cell_bg(cells3[3], LIGHT_GREY)
    score_val  = row_data.get('Patent_Interest_Score', '—')
    score_text = f"{score_val}" if score_val == '—' else f"{float(score_val):.1f}"
    add_label_value(cells3[3], 'Interest Score', score_text)

    # Row 5: Why Interesting
    add_full_width_row(table, 'WHY INTERESTING',
                       row_data.get('Interest_Reason', '—'),
                       label_hex=LABEL_HEX, value_hex='F8FBFF',
                       label_size=7, value_size=8)

    # Row 6: Abstract (budgeted to keep the card within ~one page)
    add_full_width_row(table, 'ABSTRACT',
                       _budget_card_text(row_data.get('Abstract', '—'),
                                         SECTION5_ABSTRACT_MAX_CHARS),
                       label_hex=LABEL_HEX, value_hex=WHITE,
                       label_size=7, value_size=8)

    # Row 7: Independent Claims (budgeted — this field is the main cause
    # of multi-page cards; a single patent can carry >12k chars of claims)
    add_full_width_row(table, 'INDEPENDENT CLAIMS',
                   _budget_card_text(row_data.get('Independent Claims', '—'),
                                     SECTION5_CLAIMS_MAX_CHARS),
                   label_hex=LABEL_HEX, value_hex='FAFAFA',
                   label_size=7, value_size=8)

    # Row 8: Other publications of the same invention family, if any
    member_count = int(row_data.get('Family_Member_Count', 1) or 1)
    if member_count > 1:
        other_members = [
            m for m in str(row_data.get('Family_Member_Records', '')).split(' | ')
            if m and m != record_no
        ]
        add_full_width_row(
            table, 'ALSO PUBLISHED AS',
            f"{', '.join(other_members)} ({row_data.get('Family_Member_Countries', '—')})",
            label_hex=LABEL_HEX, value_hex='F8FBFF',
            label_size=7, value_size=8
        )

    # Keep each card's rows from breaking across a page boundary so a
    # card stays visually intact and never runs over more than one page.
    _prevent_card_row_splits(table)

    # Insert extracted PDF figure snapshots below the card (notable only).
    # Drawings for notable patents intentionally follow on subsequent
    # pages — the one-page rule applies to the card table itself.
    add_pdf_figure_snapshots_to_doc(doc, record_no)

    doc.add_paragraph()


add_heading(doc, '5. Full Patent List')
doc.add_paragraph(
    f"One card per unique invention (patent family) — {len(df)} families from "
    f"{TOTAL_PUBLICATIONS} publication records this month — ordered by publication date. "
    f"Where a family has more than one publication (e.g. filed in several jurisdictions), "
    f"the card shows the representative record plus all other publications in that family. "
    f"PDF drawing pages and the downloaded-PDF-folder link are only provided for the "
    f"{len(NOTABLE_RECORD_NUMBERS)} notable patents highlighted in Section 4 — at current monthly "
    f"volume only their PDFs are bulk-downloaded. Other cards are complete as text but unlinked. "
    f"Abstract and claims are trimmed so each card stays within a page; full text is in the classified Excel."
).runs[0].font.size = Pt(10)

patent_df = df.sort_values('Publication/Issue Date', ascending=True).reset_index(drop=True)
for i, (_, row) in enumerate(patent_df.iterrows()):
    add_patent_card(doc, row, card_number=i + 1)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(patent_df)} cards rendered...")

print(f"✔ All {len(patent_df)} patent cards added.")


# ════════════════════════════════════════════════════════════════
# SAVE REPORT
# ════════════════════════════════════════════════════════════════
report_filename = (f"Patent_Report_{MONTH_NAME.replace(' ', '_')}_"
                   f"{datetime.now().strftime('%Y%m%d')}.docx")
report_path = os.path.join(FILE_DIR, report_filename)
doc.save(report_path)
print(f"\n✔ Report saved:\n  {report_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12: DOWNLOAD PDFs FOR NOTABLE / SECTION-4 PATENTS
# Downloads the actual patent PDFs (from the export's "PDF Link" column)
# for the patents shown in Section 4 — the notable shortlist plus each
# thematic table's top-N. Saves them into Downloaded_Patent_PDFs/ named by
# Record Number, so a subsequent Run All embeds their drawing pages into
# the Section-5 cards.
#
# Run this on a machine logged into PatSeer. The URLs open the PDF
# directly in a browser, so a plain request usually works; if you get
# 401/403, paste your PatSeer session cookie into PATSEER_COOKIE below.
# ═══════════════════════════════════════════════════════════════
import time, requests
import openpyxl

# ── Settings ──────────────────────────────────────────────────
PATSEER_COOKIE   = ""      # optional: "name=value; name2=value2" from browser devtools if downloads 401/403
RATE_LIMIT_SECS  = 1.0     # polite pause between downloads
DOWNLOAD_SCOPE   = "section4"   # "section4" (all Sec-4 tables) | "notable25" (shortlist only) | "all"

pdf_out_dir = os.path.join(FILE_DIR, "Downloaded_Patent_PDFs")
os.makedirs(pdf_out_dir, exist_ok=True)

# ── Which records to download ─────────────────────────────────
def _topn_records(mask):
    return set(
        df[mask].sort_values(['Patent_Interest_Score', 'Novelty_Index'], ascending=False)
                .head(SECTION4_MAX_ROWS_PER_TABLE)['Record Number'].astype(str)
    )

if DOWNLOAD_SCOPE == "all":
    target_records = set(df['Record Number'].astype(str))
elif DOWNLOAD_SCOPE == "notable25":
    target_records = set(NOTABLE_RECORD_NUMBERS)
else:  # "section4": union of everything visible in Section 4
    target_records = (
        set(NOTABLE_RECORD_NUMBERS)
        | _topn_records(df['PT_Count'] >= 3)
        | _topn_records(df['Contains_AI_ML'])
        | _topn_records(df['Performance_Targets'].str.contains('Sustainability', na=False))
        | _topn_records(df.apply(_is_smart_connected, axis=1))
    )

# ── Real URLs live in the cell HYPERLINK target, not the "PDF Link" text ──
wb = openpyxl.load_workbook(FILE_PATH)
ws = wb.active
hdr = [c.value for c in ws[1]]
rec_col = hdr.index("Record Number") + 1
pdf_col = hdr.index("PDF Link") + 1
url_by_record = {}
for r in range(2, ws.max_row + 1):
    rec = ws.cell(row=r, column=rec_col).value
    cell = ws.cell(row=r, column=pdf_col)
    tgt = cell.hyperlink.target if cell.hyperlink else None
    if rec and tgt:
        url_by_record[str(rec).strip()] = tgt

to_download = {r: url_by_record[r] for r in target_records if r in url_by_record}
missing_url = sorted(target_records - set(url_by_record))

def _safe(rec):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(rec))[:80]

sess = requests.Session()
sess.headers.update({"User-Agent": "Mozilla/5.0"})
if PATSEER_COOKIE:
    sess.headers.update({"Cookie": PATSEER_COOKIE})

print(f"Scope '{DOWNLOAD_SCOPE}': {len(target_records)} records | {len(to_download)} have PDF URLs "
      f"| {len(missing_url)} missing a URL")
print(f"Saving to: {pdf_out_dir}\n")

ok, skipped, failed = 0, 0, []
for i, (rec, url) in enumerate(sorted(to_download.items()), 1):
    dest = os.path.join(pdf_out_dir, f"{_safe(rec)}.pdf")
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        skipped += 1
        continue
    try:
        resp = sess.get(url, timeout=60)
        if resp.status_code == 200 and resp.content[:5] == b"%PDF-":
            with open(dest, "wb") as f:
                f.write(resp.content)
            ok += 1
            print(f"  [{i}/{len(to_download)}] OK  {rec}")
        else:
            failed.append((rec, f"HTTP {resp.status_code}, not a PDF (len={len(resp.content)})"))
            print(f"  [{i}/{len(to_download)}] --  {rec}  HTTP {resp.status_code}, not a PDF")
    except Exception as e:
        failed.append((rec, str(e)))
        print(f"  [{i}/{len(to_download)}] --  {rec}  {type(e).__name__}: {e}")
    time.sleep(RATE_LIMIT_SECS)

print(f"\nDownloaded: {ok} | Already present: {skipped} | Failed: {len(failed)} | No URL: {len(missing_url)}")
if failed:
    print("\nFailures (if these are all HTTP 401/403, set PATSEER_COOKIE and re-run):")
    for rec, why in failed[:20]:
        print(f"  {rec}: {why}")
print("\nRe-run the whole notebook after this to embed the downloaded drawings into Section 5.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# HEATMAPS FROM IMAGE-TRANSCRIBED VALUES
# Monthly + Cumulative
# Viridis + automatic annotation coloring
# ═══════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as path_effects
from matplotlib.colors import Normalize

# ---------------------------------------------------------------------------
# 0. Save directory
# ---------------------------------------------------------------------------

if "FILE_DIR" not in globals():
    FILE_DIR = os.getcwd()

print("=" * 90)
print("MONTHLY + CUMULATIVE HEATMAPS FROM IMAGE VALUES")
print("=" * 90)

# ---------------------------------------------------------------------------
# 1. Common row/column order
#    Using same order for both charts for easy comparison
# ---------------------------------------------------------------------------

# NOTE: taxonomy fix (see Cell 5) renamed/moved a couple of categories —
# 'Zero_Degree_Belt' is now a Technology Lever (was 'Zero_Degree' under
# Performance Targets). This hardcoded transcription cell was not updated
# for it since it will be replaced by the automated cross-month master
# dataset (tracked separately), not patched further by hand.
TECH_ORDER = [
    "Compound",
    "Tread_Design",
    "Structure",
    "Process",
    "Smart",
    "Electrical_Functionality",
]

PT_ORDER = [
    "Low_RR",
    "Wet_Grip",
    "Snow_Ice",
    "Wear_Mileage",
    "Cut_Chip",
    "NVH",
    "Durability",
    "HP",
    "Smart_Connected",
    "Sustainability",
    "Zero_Degree",
    "Electrical_Properties",
]

# ---------------------------------------------------------------------------
# 2. CUMULATIVE VALUES
#    Transcribed from the first image
# ---------------------------------------------------------------------------

cumulative_values = pd.DataFrame(
    [
        [42, 14, 1, 18, 1, 8, 28, 3, 13, 19, 1, 2],  # Compound
        [28,  7, 5,  9, 0,10, 24, 2,  9,  6, 1, 1],  # Tread_Design
        [51, 15, 3, 21, 1,19, 67, 5, 38, 32,13, 3],  # Structure
        [ 7,  1, 1,  2, 0, 3, 10, 0,  9,  7, 1, 1],  # Process
        [ 2,  1, 0,  0, 0, 4,  3, 0, 46,  2, 0, 1],  # Smart
        [ 0,  0, 0,  0, 0, 0,  1, 0,  1,  0, 0, 2],  # Electrical_Functionality
    ],
    index=TECH_ORDER,
    columns=PT_ORDER
)

# ---------------------------------------------------------------------------
# 3. MONTHLY VALUES
#    Transcribed from the second image
#    Missing columns in the image (HP, Zero_Degree) are filled with 0
# ---------------------------------------------------------------------------

monthly_values = pd.DataFrame(
    [
        [16, 7, 3, 5, 1, 5, 13, 0, 1, 1, 0, 2],  # Compound
        [ 8, 2, 1, 5, 0, 6,  5, 0, 1, 1, 0, 1],  # Tread_Design
        [18, 6, 4, 7, 1, 7, 16, 0, 2, 1, 0, 3],  # Structure
        [ 8, 3, 1, 3, 1, 1,  3, 0, 1, 0, 0, 1],  # Process
        [ 0, 0, 0, 0, 0, 0,  0, 0, 3, 0, 0, 1],  # Smart
        [ 0, 0, 0, 0, 0, 0,  0, 0, 0, 0, 0, 1],  # Electrical_Functionality
    ],
    index=TECH_ORDER,
    columns=PT_ORDER
)

# ---------------------------------------------------------------------------
# 4. Helper functions
# ---------------------------------------------------------------------------

def relative_luminance(rgba):
    r, g, b = rgba[:3]
    return 0.299 * r + 0.587 * g + 0.114 * b


def draw_heatmap(ax, matrix, title, drop_all_zero_cols=False, drop_all_zero_rows=False):
    """
    Draw heatmap with:
    - viridis colormap
    - masked zero cells
    - automatic annotation color
    """

    matrix_plot = matrix.copy()

    if drop_all_zero_rows:
        matrix_plot = matrix_plot.loc[matrix_plot.sum(axis=1) > 0]

    if drop_all_zero_cols:
        matrix_plot = matrix_plot.loc[:, matrix_plot.sum(axis=0) > 0]

    if matrix_plot.empty:
        ax.axis("off")
        ax.text(0.5, 0.5, "No data available", ha="center", va="center",
                fontsize=14, fontweight="bold")
        return

    max_count = int(matrix_plot.values.max())
    if max_count == 0:
        ax.axis("off")
        ax.text(0.5, 0.5, "All values are zero", ha="center", va="center",
                fontsize=14, fontweight="bold")
        return

    cmap = plt.colormaps["viridis"].copy()
    zero_cell_color = "#440154"   # viridis low-end purple
    cmap.set_bad(color=zero_cell_color)

    plot_data = matrix_plot.replace(0, np.nan)
    mask = plot_data.isna()
    norm = Normalize(vmin=1, vmax=max_count)

    sns.heatmap(
        plot_data,
        annot=False,
        fmt="",
        cmap=cmap,
        norm=norm,
        mask=mask,
        linewidths=0.7,
        linecolor="#d9d9d9",
        ax=ax,
        cbar=True,
        cbar_kws={"label": "Patent Count", "shrink": 0.85, "pad": 0.02}
    )

    ax.set_facecolor(zero_cell_color)

    # Manual annotations with auto text color
    for i in range(matrix_plot.shape[0]):
        for j in range(matrix_plot.shape[1]):
            value = int(matrix_plot.iloc[i, j])

            if value == 0:
                continue

            rgba = cmap(norm(value))
            lum = relative_luminance(rgba)

            if lum > 0.53:
                txt_color = "black"
                outline_color = "white"
            else:
                txt_color = "white"
                outline_color = "black"

            txt = ax.text(
                j + 0.5,
                i + 0.5,
                str(value),
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color=txt_color
            )

            txt.set_path_effects([
                path_effects.Stroke(linewidth=1.8, foreground=outline_color),
                path_effects.Normal()
            ])

    ax.set_title(title, fontsize=16, fontweight="bold", pad=14)
    ax.set_xlabel("Performance Targets", fontsize=13, fontweight="bold", labelpad=10)
    ax.set_ylabel("Technology Levers", fontsize=13, fontweight="bold", labelpad=10)

    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#2b2440")
        spine.set_linewidth(1.1)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("Patent Count", fontsize=11, fontweight="bold")


# ---------------------------------------------------------------------------
# 5. Plot Monthly Heatmap
#    Here I drop zero-only columns/rows so it looks close to your screenshot
# ---------------------------------------------------------------------------

fig1, ax1 = plt.subplots(figsize=(14, 7), facecolor="white")
draw_heatmap(
    ax1,
    monthly_values,
    title="Technology Levers vs Performance Targets\nPatent Co-occurrence Count",
    drop_all_zero_cols=True,
    drop_all_zero_rows=False
)
plt.tight_layout()

monthly_path = os.path.join(FILE_DIR, "monthly_heatmap_viridis_from_image.png")
plt.savefig(monthly_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"✔ Monthly heatmap saved: {monthly_path}")

# ---------------------------------------------------------------------------
# 6. Plot Cumulative Heatmap
# ---------------------------------------------------------------------------

fig2, ax2 = plt.subplots(figsize=(16, 7), facecolor="white")
draw_heatmap(
    ax2,
    cumulative_values,
    title="Cumulative Technology Levers vs Performance Targets\n(Jan 2026 - Present)",
    drop_all_zero_cols=False,
    drop_all_zero_rows=False
)
plt.tight_layout()

cumulative_path = os.path.join(FILE_DIR, "cumulative_heatmap_viridis_from_image.png")
plt.savefig(cumulative_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"✔ Cumulative heatmap saved: {cumulative_path}")

# ---------------------------------------------------------------------------
# 7. Combined Figure
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(
    2, 1,
    figsize=(16, 14),
    facecolor="white"
)

draw_heatmap(
    axes[0],
    cumulative_values,
    title="Cumulative Technology Levers vs Performance Targets\n(Jan 2026 - Present)",
    drop_all_zero_cols=False,
    drop_all_zero_rows=False
)

draw_heatmap(
    axes[1],
    monthly_values,
    title="Technology Levers vs Performance Targets\nPatent Co-occurrence Count",
    drop_all_zero_cols=True,
    drop_all_zero_rows=False
)

plt.tight_layout()

combined_path = os.path.join(FILE_DIR, "combined_monthly_cumulative_heatmaps_viridis.png")
plt.savefig(combined_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"✔ Combined heatmap saved: {combined_path}")

# ---------------------------------------------------------------------------
# 8. Print matrices for checking
# ---------------------------------------------------------------------------

print("\n" + "=" * 90)
print("CUMULATIVE MATRIX")
print("=" * 90)
print(cumulative_values)

print("\n" + "=" * 90)
print("MONTHLY MATRIX")
print("=" * 90)
print(monthly_values)

print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)
print(f"Monthly total co-occurrences    : {int(monthly_values.values.sum())}")
print(f"Cumulative total co-occurrences : {int(cumulative_values.values.sum())}")
print(f"Monthly max cell value          : {int(monthly_values.values.max())}")
print(f"Cumulative max cell value       : {int(cumulative_values.values.max())}")
print("=" * 90)